# GameTheory 3b — Chambres, murs et chemins : la géométrie ordinale des jeux 2×2 (C# / .NET)

> **Deuxième volet de la chaîne 03** — précédent : [GameTheory-03-Topology2x2-CSharp](GameTheory-03-Topology2x2-CSharp.ipynb) (les 576 jeux stricts et la grammaire des swaps) ; suivants : [Dynamics-10-Joueur-LLM-Python](Dynamics/Dynamics-10-Joueur-LLM-Python.ipynb) (la table périodique appliquée à un LLM), [GameTheory-03d-Deux-Especes-de-Fleches-Python](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) (quand un swap préserve les équilibres) et [GameTheory-03e-Meta-Actions-Tarifees-Python](GameTheory-03e-Meta-Actions-Tarifees-Python.ipynb) (le capstone).

Ce carnet décrit **l'espace** dans lequel les transformations de GT-3 se déplacent, puis **sa métrique** : les strates de l'espace des paiements et leur incidence (§1-§4), puis la distance de swap entre deux jeux et les chemins qui la réalisent (§5-§10). Les deux moitiés se lisent dans cet ordre — une distance se mesure en murs traversés, et les murs sont ce que la première moitié construit.

**La thèse, en une phrase** : un jeu à égalités n'est pas un jeu « plus flou » ou « plus général » qu'un jeu strict — c'est un objet **de dimension inférieure**, un mur où des chambres se touchent ; et la distance de swap entre deux jeux stricts est la somme des longueurs de Coxeter de leurs deux côtés, mesurée par le nombre de murs traversés.

La lecture géométrique vient de la formalisation de **Bruns & Kimmich** (rapportée, cf. §Sources) :

- les jeux **stricts** — aucune égalité de paiements — sont les **chambres** : les composantes ouvertes de l'espace des paiements ;
- les jeux **à égalités** sont les **murs** : des strates de **codimension ≥ 1** sur lesquelles plusieurs chambres se raccordent ;
- une transformation élémentaire n'est pas un saut abstrait d'un jeu à l'autre : elle **traverse une face**.

```
chambre  →  mur  →  chambre voisine
```

Un « objet botanique » — un jeu à égalités depuis lequel plusieurs prolongements sont possibles — n'est donc **pas** une définition assouplie : c'est **habiter le bord**. La différence entre les deux notions est la différence entre *relâcher un axiome* et *se placer sur une strate de codimension 1*.

**Objectifs d'apprentissage** — à la fin de ce carnet, vous savez :

1. énumérer les strates de l'espace des paiements et les classer par codimension (75 ordres faibles par joueur, 5 625 jeux à rangs) ;
2. mesurer l'incidence chambre-mur et lire le graphe des chambres (connexité, diamètre, distribution des distances) ;
3. calculer une distance de swap par parcours en largeur, et énumérer les plus courts chemins entre deux jeux ;
4. vérifier le théorème de décomposition $d(G,H) = \ell(r_G,r_H) + \ell(c_G,c_H)$ par énumération exhaustive, et distinguer un chemin **valide** d'un chemin **minimal** — le premier se certifie, le second se mesure.

**Prérequis** : [GameTheory-03-Topology2x2-CSharp](GameTheory-03-Topology2x2-CSharp.ipynb) — les 576 jeux stricts et les six générateurs $R_{12}/R_{23}/R_{34}/C_{12}/C_{23}/C_{34}$. .NET 8 (C# 12), BCL seule, aucune dépendance externe.

**Durée estimée** : 1 h 30 de lecture active, plus les six exercices.

**Plan** : §1 les strates classées par codimension · §2 l'incidence chambre-mur et le graphe des chambres · §3 traverser un mur, puis relire la grammaire des swaps en longueurs de Coxeter · §4 *make tie* / *break tie* et trois archétypes · §5 l'espace des 576 jeux et ses générateurs · §6 la distance d'un jeu à l'autre — BFS depuis le Dilemme, tableau des distances aux archétypes, les trois cousins à distance 2 · §7 la distribution des distances — sphères emboîtées, symétrie parfaite, convolution des distributions d'inversions de $S_4$ · §8 le théorème de décomposition — la distance est la somme des longueurs de Coxeter des deux côtés, vérifié par énumération exhaustive sur les 331 776 paires · §9 les multiplicités des plus courts chemins — de 2 (commutation) à 236 544 (l'anti-Dilemme) · §10 générateur contre certificat — le miroir .NET du vérificateur Lean, valide ≠ minimal, puis le module `Swaps/Basic.lean` lui-même, cité aux lignes mesurées · **Exercices** (6) — énumérer les murs par codimension, exhiber un triplet chambre-mur-chambre, make tie / break tie vers les archétypes, puis distance par sphères, diamètre depuis une autre source et certificat d'un chemin fourni.

**Conventions** : un jeu 2×2 ordinal = deux tables de paiements (Ligne, Colonne), chacune un 4-uplet $(r_{11}, r_{12}, r_{21}, r_{22})$ de valeurs ; un jeu strict = chaque table est une permutation de $\{1,2,3,4\}$.


## 0. Configuration

Notebook **pur BCL** (`System`, `System.Collections.Generic`, `System.Linq`, `System.Globalization`) — aucune dépendance, aucun paquet NuGet, aucun accès réseau. Comme GT-21, tout ce qui est affirmé ici est **calculé dans ce notebook** ; ce qui vient de la littérature sans dérivation locale est **rapporté comme dette** (§Sources).

In [1]:
// GameTheory 3b : chambres, murs et chemins -- BCL pur
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

Console.WriteLine("GameTheory 3b : chambres, murs et chemins");
Console.WriteLine("Representation : jeu = (table_ligne, table_colonne), tables = ordres faibles canoniques");


GameTheory 3b : chambres, murs et chemins


Representation : jeu = (table_ligne, table_colonne), tables = ordres faibles canoniques


## 1. L'espace des paiements et ses strates

### 1.1 Ordres faibles : la clé canonique

L'espace des paiements d'un joueur est $\mathbb{R}^4$ (une coordonnée par case de sa table 2×2). Ce qui compte pour un jeu ordinal n'est pas la valeur numérique d'une coordonnée mais son **rang** relativement aux trois autres. Un point de cet espace code donc un **ordre faible** (*weak order*) sur les 4 cases : soit un ordre total (les 4 valeurs distinctes — une chambre), soit un ordre avec des ex æquo (un mur).

La représentation canonique : un 4-uplet $t$ où $t_i = 1 + (\text{nombre de valeurs distinctes strictement inférieures à } t_i)$. Ainsi $(2,2,4,5)$, $(1,1,3,4)$ et $(7,7,8,9)$ codent le même ordre faible — deux cases liées en bas, deux cases liées au-dessus — et partagent la clé canonique $(1,1,2,3)$.

Le discret $\{1,2,3,4\}$ de GT-3/GT-21 est le cas particulier sans ex æquo : les 24 permutations de $(1,2,3,4)$.


In [2]:
// Section 1.1 : enumeration des ordres faibles sur 4 cases

int[] Canonique(int[] t)
{
    var distinctes = t.Distinct().OrderBy(v => v).ToArray();
    var rang = new int[t.Length];
    for (int i = 0; i < t.Length; i++)
    {
        rang[i] = Array.IndexOf(distinctes, t[i]) + 1;
    }
    return rang;
}

int CompareTuples(int[] a, int[] b)
{
    for (int i = 0; i < Math.Min(a.Length, b.Length); i++)
    {
        int c = a[i].CompareTo(b[i]);
        if (c != 0) return c;
    }
    return a.Length.CompareTo(b.Length);
}

string Format(int[] t) => "(" + string.Join(", ", t) + ")";

// Enumeration : tous les 4-uplets de {1..4} (256), projetes sur leur cle canonique
var quadruplets = new List<int[]>();
for (int a = 1; a <= 4; a++)
{
    for (int b = 1; b <= 4; b++)
    {
        for (int c = 1; c <= 4; c++)
        {
            for (int d = 1; d <= 4; d++)
            {
                quadruplets.Add(new int[] { a, b, c, d });
            }
        }
    }
}

var vues = new HashSet<string>();
var ordres_faibles = new List<int[]>();
foreach (var q in quadruplets)
{
    var cle = Canonique(q);
    if (vues.Add(string.Join(",", cle))) ordres_faibles.Add(cle);
}
ordres_faibles.Sort(CompareTuples);

int stricts = ordres_faibles.Count(t => t.Distinct().Count() == 4);
Console.WriteLine("Ordres faibles sur 4 cases : " + ordres_faibles.Count);
Console.WriteLine("  stricts (chambres potentielles) : " + stricts);
Console.WriteLine("  avec ex aequo (murs potentiels)  : " + (ordres_faibles.Count - stricts));


Ordres faibles sur 4 cases : 75


  stricts (chambres potentielles) : 24


  avec ex aequo (murs potentiels)  : 51


**Lecture de la sortie committée** : **75 ordres faibles** — c'est le *nombre de Fubini* $a(4)$, le compte des partitions ordonnées d'un ensemble à 4 éléments. Ils se partagent en **24 ordres totaux** (les permutations — les chambres au sens strict, inchangées depuis GT-3) et **51 ordres avec ex æquo** (les murs, par joueur).

Ce simple compte situe le changement de regard : le discret de GT-3 ($24^2 = 576$ jeux stricts) vivait dans un espace où les égalités étaient **exclues par construction**. En admettant les ex æquo dans la représentation, l'espace passe de $576$ à $75^2 = 5625$ jeux à rangs — et ces nouveaux points ne sont pas des « jeux imprécis » : ce sont des **strates**. Reste à leur donner une dimension.


### 1.2 Codimension : compter des égalités *indépendantes*

La codimension d'une strate est le nombre d'**équations indépendantes** qui la définissent — la dimension perdue. Pour un ordre faible, ce n'est pas le nombre de *paires* égales mais le **complément du nombre de blocs** :

$$\text{codim}(t) = 4 - \#\text{blocs}(t).$$

Un bloc de taille 3 (triple égalité) porte **3 paires** égales mais seulement **2 équations indépendantes** ($x_a = x_b$ et $x_b = x_c$ suffisent, la troisième s'en déduit). Confondre paires et équations est l'erreur classique — le code suivant les compte séparément pour montrer qu'elles divergent.


In [3]:
// Section 1.2 : codimension vraie vs nombre de paires egales

int NbBlocs(int[] t) => t.Distinct().Count();

int NbPairesEgales(int[] t) => t.GroupBy(v => v).Sum(g => g.Count() * (g.Count() - 1) / 2);

int Codim(int[] t) => 4 - NbBlocs(t);

int[] SignatureBlocs(int[] t) =>
    t.GroupBy(v => v).Select(g => g.Count()).OrderByDescending(n => n).ToArray();

var parType = new Dictionary<string, List<int[]>>();
var signatures = new List<int[]>();
foreach (var t in ordres_faibles)
{
    var sig = SignatureBlocs(t);
    var cle = string.Join(",", sig);
    if (!parType.TryGetValue(cle, out var lot))
    {
        lot = new List<int[]>();
        parType[cle] = lot;
        signatures.Add(sig);
    }
    lot.Add(t);
}
// tri decroissant des signatures, comme le reverse=True du carnet Python
signatures.Sort((x, y) => CompareTuples(y, x));

Console.WriteLine("Type de blocs           | codim | paires | exemples");
Console.WriteLine(new string('-', 64));
foreach (var sig in signatures)
{
    var lot = parType[string.Join(",", sig)];
    int pairesMoy = lot.Sum(NbPairesEgales) / lot.Count;
    string exemples = Format(lot[0]) + (lot.Count > 1 ? ", " + Format(lot[1]) : "");
    Console.WriteLine($"blocs {Format(sig),-10} ({lot.Count,2} ordres) |   {4 - sig.Length}   |   {pairesMoy}    | {exemples}");
}
Console.WriteLine();

var distCodim = ordres_faibles.GroupBy(Codim).ToDictionary(g => g.Key, g => g.Count());
Console.WriteLine("Distribution par codimension (par joueur) : {" +
    string.Join(", ", distCodim.OrderBy(kv => kv.Key).Select(kv => kv.Key + ": " + kv.Value)) + "}");
Console.WriteLine("Verification de la somme : 24 + 36 + 14 + 1 = " + ordres_faibles.Count + " = 75 ordres faibles");


Type de blocs           | codim | paires | exemples


----------------------------------------------------------------


blocs (4)        ( 1 ordres) |   3   |   6    | (1, 1, 1, 1)


blocs (3, 1)     ( 8 ordres) |   2   |   3    | (1, 1, 1, 2), (1, 1, 2, 1)


blocs (2, 2)     ( 6 ordres) |   2   |   2    | (1, 1, 2, 2), (1, 2, 1, 2)


blocs (2, 1, 1)  (36 ordres) |   1   |   1    | (1, 1, 2, 3), (1, 1, 3, 2)


blocs (1, 1, 1, 1) (24 ordres) |   0   |   0    | (1, 2, 3, 4), (1, 2, 4, 3)


Distribution par codimension (par joueur) : {0: 24, 1: 36, 2: 14, 3: 1}


Verification de la somme : 24 + 36 + 14 + 1 = 75 = 75 ordres faibles


**Lecture de la sortie committée** : la taxonomy par taille de blocs croise les deux mesures.

| blocs | codim | paires égales | compte |
|---|---|---|---|
| $(1,1,1,1)$ | 0 | 0 | 24 — **les chambres** |
| $(2,1,1)$ | 1 | 1 | 36 — **les murs simples** |
| $(2,2)$ | 2 | 2 | 6 |
| $(3,1)$ | **2** | **3** | 8 |
| $(4)$ | **3** | **6** | 1 |

Les deux lignes en **gras** portent la leçon : le triple ex æquo (les 8 cas $(3,1)$) **diverge** — 3 paires mais codimension 2 — et le quadruple ex æquo porte 6 paires pour codimension 3. La codimension est l'auche correcte : elle mesure la **dimension perdue**, pas le nombre de coïncidences.

L'image botanique se précise : les murs de codimension 1 sont des **cloisons** (36 par joueur), ceux de codimension 2 des **arêtes** où les cloisons se rencontrent (6+8 = 14), celui de codimension 3 un **sommet** (1). Un objet botanique n'est pas « un jeu imprécis » : c'est un point de l'espace des paiements situé **sur** cette architecture.


### 1.3 Les 5 625 jeux à rangs, par codimension

Un jeu complet = une paire d'ordres faibles (Ligne, Colonne). La codimension du jeu est la somme des deux : un mur de Ligne dans une chambre de Colonne est un mur de codim 1 ; deux murs simultanés donnent codim 2 ; etc.


In [4]:
// Section 1.3 : distribution des jeux par codimension totale

var jeux = (from r in ordres_faibles
            from c in ordres_faibles
            select (r, c)).ToList();

var distJeux = jeux
    .GroupBy(g => Codim(g.Item1) + Codim(g.Item2))
    .ToDictionary(g => g.Key, g => g.Count());

Console.WriteLine("Jeux a rangs (paires d'ordres faibles) : " + jeux.Count);
Console.WriteLine();
Console.WriteLine("codim |  jeux  | remarque");
Console.WriteLine(new string('-', 56));
var remarques = new Dictionary<int, string>
{
    { 0, "les 576 chambres strictes (univers GT-3 / GT-21)" },
    { 1, "murs simples : un seul ex aequo, d'un seul cote" },
    { 2, "murs doubles : deux ex aequo (eventuellement croises)" },
    { 6, "les deux tables entierement plates (jeu unique)" }
};
foreach (var kv in distJeux.OrderBy(kv => kv.Key))
{
    string remarque = remarques.TryGetValue(kv.Key, out var txt) ? txt : "";
    Console.WriteLine($"  {kv.Key}   | {kv.Value,5}  | {remarque}");
}
Console.WriteLine();
Console.WriteLine("Total : " + jeux.Count + " = 75 x 75 attendu");
Console.WriteLine("Chambres strictes : " + distJeux[0] + " -- coherentes avec GT-3 (576) et GT-21");


Jeux a rangs (paires d'ordres faibles) : 5625


codim |  jeux  | remarque


--------------------------------------------------------


  0   |   576  | les 576 chambres strictes (univers GT-3 / GT-21)


  1   |  1728  | murs simples : un seul ex aequo, d'un seul cote


  2   |  1968  | murs doubles : deux ex aequo (eventuellement croises)


... (2 outputs left out)

  5   |    28  | 


  6   |     1  | les deux tables entierement plates (jeu unique)


Total : 5625 = 75 x 75 attendu


Chambres strictes : 576 -- coherentes avec GT-3 (576) et GT-21


**Lecture de la sortie committée** : l'univers des jeux à rangs compte **5 625** états. Les **576 chambres strictes** — tout l'univers de GT-3 et de GT-21 — n'en occupent que **10,2 %**. Les **1 728 jeux de codimension 1** sont les murs simples ; **1 968** de codim 2 ; le reste s'étage jusqu'au jeu totalement plat (codim 6), unique, où chaque joueur est indifférent entre les quatre cases.

Autrement dit : le discret dans lequel la série a travaillé jusqu'ici était le **squelette ouvert** d'un édifice dont les murs, arêtes et sommets — 89,8 % des états à rangs — n'avaient pas de nom. La suite leur donne un rôle structurel : ce sont eux qui **raccordent** les chambres.


## 2. L'incidence chambre-mur

### 2.1 Chaque mur simple sépare exactement deux chambres

Un mur de codimension 1 (un seul ex æquo, d'un seul côté) est une cloison : la briser dans un sens ou dans l'autre donne **exactement deux** chambres. Réciproquement, depuis une chambre, quels murs touche-t-on ? Pas tous : pour que deux cases puissent **se rencontrer**, il faut qu'aucune autre valeur ne soit strictement comprise entre les leurs — sinon le mouvement croiserait d'autres murs d'abord. Les facettes d'une chambre correspondent aux paires de **valeurs adjacentes** dans son ordre.

Le code suivant mesure l'incidence dans les deux directions.


In [5]:
// Section 2.1 : incidence double-face chambre <-> mur

var chambres = ordres_faibles.Where(t => Codim(t) == 0).ToList();      // 24 permutations
var mursSimples = ordres_faibles.Where(t => Codim(t) == 1).ToList();  // 36

int[] Rangs(int[] positions)
{
    var rk = new int[4];
    for (int k = 0; k < positions.Length; k++) rk[positions[k]] = k + 1;
    return rk;
}

(int[] A, int[] B) BriserLeTie(int[] t)
{
    // Les deux chambres strictes adjacentes au mur simple t (cote de t seul).
    var multiplicites = t.GroupBy(v => v).ToDictionary(g => g.Key, g => g.Count());
    int vTie = multiplicites.First(kv => kv.Value == 2).Key;
    var lies = Enumerable.Range(0, 4).Where(p => t[p] == vTie).ToArray();
    int i = lies[0], j = lies[1];
    var below = Enumerable.Range(0, 4).Where(p => t[p] < vTie).OrderBy(p => t[p]).ToArray();
    var above = Enumerable.Range(0, 4).Where(p => t[p] > vTie).OrderBy(p => t[p]).ToArray();
    // chambre A : i passe devant j ; chambre B : j devant i -- au NIVEAU du tie
    var A = Rangs(below.Concat(new[] { i, j }).Concat(above).ToArray());
    var B = Rangs(below.Concat(new[] { j, i }).Concat(above).ToArray());
    return (A, B);
}

// mur -> chambres
var incidencesMur = new Dictionary<string, (int[] Mur, int[] A, int[] B)>();
foreach (var t in mursSimples)
{
    var (a, b) = BriserLeTie(t);
    incidencesMur[string.Join(",", t)] = (t, a, b);
}
bool toutesDistinctes = incidencesMur.Values.All(p => !p.A.SequenceEqual(p.B));
Console.WriteLine("Murs simples par cote : " + mursSimples.Count);
Console.WriteLine("Chaque mur touche exactement 2 chambres distinctes : " + toutesDistinctes);

// chambre -> murs (facettes)
var facettes = new Dictionary<string, List<int[]>>();
foreach (var kv in incidencesMur)
{
    foreach (var face in new[] { kv.Value.A, kv.Value.B })
    {
        string cle = string.Join(",", face);
        if (!facettes.TryGetValue(cle, out var lot))
        {
            lot = new List<int[]>();
            facettes[cle] = lot;
        }
        lot.Add(kv.Value.Mur);
    }
}
var nbFacettes = facettes.Values.GroupBy(v => v.Count).ToDictionary(g => g.Key, g => g.Count());
Console.WriteLine("Facettes par chambre  : {" +
    string.Join(", ", nbFacettes.OrderBy(kv => kv.Key).Select(kv => kv.Key + ": " + kv.Value)) + "}" +
    " (3 attendu : les 3 paires de valeurs adjacentes)");


Murs simples par cote : 36


Chaque mur touche exactement 2 chambres distinctes : True


Facettes par chambre  : {3: 24} (3 attendu : les 3 paires de valeurs adjacentes)


**Lecture de la sortie committée** : la double incidence est **exacte**.

- **Mur → chambres** : chacun des 36 murs simples d'un côté touche **exactement 2 chambres** distinctes — les deux résolutions de son ex æquo. Une cloison n'a jamais trois pièces de chaque côté.
- **Chambre → murs** : chacune des 24 chambres a **exactement 3 facettes** — les 3 paires de valeurs *adjacentes* de son ordre. Une paire non adjacente (les valeurs 1 et 3, avec 2 entre elles) ne définit **pas** une facette : pour amener 1 et 3 à l'égalité, il faut d'abord traverser le mur où 1 rencontre 2.

C'est la structure d'**arrangement d'hyperplans** de l'espace des paiements (les murs étant les hyperplans $x_i = x_j$) — calculée ici sans en invoquer la théorie : les comptes parlent d'eux-mêmes. Au niveau du jeu complet (deux côtés), chaque chambre a donc $3 + 3 = 6$ facettes, et le nombre de murs simples du jeu vaut $2 \times 36 \times 24 = 1728$.

**Remarque de parité avec GT-21** : le théorème « le mur ne compte que s'il est habité » de GT-21 portait sur les *meilleures réponses* (quelles stratégies habitent la colonne traversée) ; ici l'habitation est *géométrique* — quelles chambres touchent le mur. Les deux lectures se complètent : GT-21 dit quand un swap **change** les équilibres, la lecture géométrique dit **où** le swap se déplace.


### 2.2 Le graphe des chambres : connexité et diamètre

Déclarer que les murs « raccordent » les chambres donne un graphe : les sommets sont les 576 chambres, une arête par mur simple traversé. La bonne nouvelle pour la grammaire des swaps : **ce graphe est connexe** — n'importe quel jeu strict est atteignable depuis n'importe quel autre en traversant des murs un à un.


In [6]:
// Section 2.2 : BFS sur le graphe des chambres

int[] SwapValeursAdjacentes(int[] t, int k)
{
    // Echange les cases portant les valeurs k et k+1 (traversee de la facette k<->k+1).
    int pk = Array.IndexOf(t, k);
    int pk1 = Array.IndexOf(t, k + 1);
    var l = (int[])t.Clone();
    l[pk] = t[pk1];
    l[pk1] = t[pk];
    return l;
}

string Cle(int[] t) => string.Join(",", t);

var adjacences = new Dictionary<string, List<int[]>>();
foreach (var t in chambres)
{
    adjacences[Cle(t)] = new List<int[]>
    {
        SwapValeursAdjacentes(t, 1),
        SwapValeursAdjacentes(t, 2),
        SwapValeursAdjacentes(t, 3)
    };
}

// BFS cote 1 joueur, depuis (1,2,3,4)
var depart = new int[] { 1, 2, 3, 4 };
var dist = new Dictionary<string, int> { { Cle(depart), 0 } };
var file = new Queue<string>();
file.Enqueue(Cle(depart));
while (file.Count > 0)
{
    string u = file.Dequeue();
    foreach (var v in adjacences[u])
    {
        string cv = Cle(v);
        if (!dist.ContainsKey(cv))
        {
            dist[cv] = dist[u] + 1;
            file.Enqueue(cv);
        }
    }
}

string FormatDistributions(IEnumerable<int> valeurs)
{
    var regroupes = valeurs.GroupBy(x => x).OrderBy(g => g.Key).ToList();
    return "{" + string.Join(", ", regroupes.Select(g => g.Key + ": " + g.Count())) + "}";
}

Console.WriteLine("Cote 1 joueur : connexe = " + (dist.Count == 24) + " | diametre = " + dist.Values.Max());
Console.WriteLine("  repartition des distances : " + FormatDistributions(dist.Values));

// BFS jeu complet : (row, col), 576 sommets
string CleJeu(int[] row, int[] col) => Cle(row) + "|" + Cle(col);
var dep2Row = new int[] { 1, 2, 3, 4 };
var dep2Col = new int[] { 1, 2, 3, 4 };
var dist2 = new Dictionary<string, int> { { CleJeu(dep2Row, dep2Col), 0 } };
var file2 = new Queue<string>();
file2.Enqueue(CleJeu(dep2Row, dep2Col));
while (file2.Count > 0)
{
    string cu = file2.Dequeue();
    var parties = cu.Split('|');
    var ur = parties[0].Split(',').Select(int.Parse).ToArray();
    var uc = parties[1].Split(',').Select(int.Parse).ToArray();
    int dCur = dist2[cu];
    var voisinsRow = new[] { 1, 2, 3 }.Select(k => CleJeu(SwapValeursAdjacentes(ur, k), uc)).ToArray();
    var voisinsCol = new[] { 1, 2, 3 }.Select(k => CleJeu(ur, SwapValeursAdjacentes(uc, k))).ToArray();
    foreach (string cv in voisinsRow.Concat(voisinsCol))
    {
        if (!dist2.ContainsKey(cv))
        {
            dist2[cv] = dCur + 1;
            file2.Enqueue(cv);
        }
    }
}
Console.WriteLine();
Console.WriteLine("Jeu complet : connexe = " + (dist2.Count == 576) + " | diametre = " + dist2.Values.Max());
Console.WriteLine("  repartition : " + FormatDistributions(dist2.Values));
Console.WriteLine("  aretes = 576 x 6 / 2 = " + (576 * 6 / 2) + " -- a comparer aux 1728 murs simples du 1.3");


Cote 1 joueur : connexe = True | diametre = 6


  repartition des distances : {0: 1, 1: 3, 2: 5, 3: 6, 4: 5, 5: 3, 6: 1}


Jeu complet : connexe = True | diametre = 12


  repartition : {0: 1, 1: 6, 2: 19, 3: 42, 4: 71, 5: 96, 6: 106, 7: 96, 8: 71, 9: 42, 10: 19, 11: 6, 12: 1}


  aretes = 576 x 6 / 2 = 1728 -- a comparer aux 1728 murs simples du 1.3


**Lecture de la sortie committée** : le graphe des chambres est **connexe**, de diamètre **6 par côté** et **12 pour le jeu complet** — le pire cas étant l'ordre exactement renversé des deux tables ($6 + 6$ inversions).

Deux régularités lisibles dans la sortie :

- la distribution des distances d'un côté, $\{0{:}1, 1{:}3, 2{:}5, 3{:}6, 4{:}5, 5{:}3, 6{:}1\}$, est **symétrique** : le mot le plus long (l'ordre renversé) est unique, comme le plus court (l'identité) ;
- au niveau du jeu, chaque chambre a **6 voisins** (3 facettes par côté), et le compte d'arêtes $576 \times 6 / 2 = 1728$ coïncide **exactement** avec le nombre de murs simples comptés au §1.3 — chaque mur simple **est** une arête du graphe, chaque arête **est** la traversée d'un mur. Le pont entre le comptage de strates (§1) et la connectivité (§2) est un double-comptage propre.

Conséquence pour la série : le discret $\{1,2,3,4\}$ de GT-3 n'était pas un artefact de commodité — c'est un **transversal** des chambres (un point par chambre), et la connexité ci-dessus garantit que la grammaire des swaps adjacents suffit à relier tout l'univers strict.


## 3. Traverser un mur : chambre → mur → chambre

### 3.1 Un cas complet, matrices à l'appui

Prenons le **Dilemme du Prisonnier** dans l'encodage ordinal de GT-21 — Ligne $(3,1,4,2)$, Colonne $(3,4,1,2)$ — et suivons le swap de valeurs adjacentes $R(3,4)$ (Ligne échange les cases portant 3 et 4). Ce swap traverse la facette où les deux cases porteuses deviennent égales : le mur $r_{11} = r_{21}$.


In [7]:
// Section 3.1 : traverser le mur r11=r21 depuis le Dilemme du Prisonnier

void AfficherJeu(string nom, int[] row, int[] col)
{
    Console.WriteLine(nom);
    Console.WriteLine($"  Ligne    | {row[0],2}  {row[1],-2}|   Colonne | {col[0],2}  {col[1],-2}|");
    Console.WriteLine($"           | {row[2],2}  {row[3],-2}|           | {col[2],2}  {col[3],-2}|");
    Console.WriteLine();
}

int[] FaireLeTie(int[] t, int k)
{
    // Coalescence des valeurs k et k+1 -> mur canonique (make_tie du 4.1, cote seul).
    return Canonique(t.Select(x => x == k + 1 ? k : x).ToArray());
}

// La chambre de depart : le PD (encodage GT-21)
var pdRow = new int[] { 3, 1, 4, 2 };
var pdCol = new int[] { 3, 4, 1, 2 };
AfficherJeu("CHAMBRE A -- Dilemme du Prisonnier (strict)", pdRow, pdCol);

// Le mur : coalescence des valeurs 3 et 4 de Ligne (cases r11 et r21)
var murRow = FaireLeTie(pdRow, 3);
AfficherJeu("MUR (codim 1) -- Ligne indifferent r11 = r21", murRow, pdCol);

// La chambre d'arrivee : resolution du tie dans l'autre sens
var chambreBRow = SwapValeursAdjacentes(pdRow, 3);
AfficherJeu("CHAMBRE B -- apres traversee (swap R(3,4))", chambreBRow, pdCol);

var (faceA, faceB) = BriserLeTie(murRow);
Console.WriteLine("Verification : mur = " + Format(murRow));
Console.WriteLine("  BriserLeTie(mur) redonne les deux faces : " + Format(faceA) + ", " + Format(faceB));
Console.WriteLine("  chambre A : " + Format(Canonique(pdRow)) + " chambre B : " + Format(Canonique(chambreBRow)));


CHAMBRE A -- Dilemme du Prisonnier (strict)


  Ligne    |  3  1 |   Colonne |  3  4 |


           |  4  2 |           |  1  2 |


MUR (codim 1) -- Ligne indifferent r11 = r21


  Ligne    |  3  1 |   Colonne |  3  4 |


           |  3  2 |           |  1  2 |


... (2 outputs left out)

           |  3  2 |           |  1  2 |


Verification : mur = (3, 1, 3, 2)


  BriserLeTie(mur) redonne les deux faces : (3, 1, 4, 2), (4, 1, 3, 2)


  chambre A : (3, 1, 4, 2) chambre B : (4, 1, 3, 2)


**Lecture de la sortie committée** : le triplet complet **chambre A → mur → chambre B** est exhibé avec les trois matrices — l'acceptation de l'exercice 2 est satisfaite *et démontrée* :

- **chambre A** : le PD strict, où $r_{21} = 4 > r_{11} = 3$ (Ligne gagne plus à dévier) ;
- **mur** : le jeu à égalité où $r_{11} = r_{21}$ — Ligne est **exactement indifférent** entre ses deux stratégies quand Colonne joue à gauche ; c'est le point de bascule où la défection cesse d'être strictement meilleure *en première colonne* ;
- **chambre B** : l'ordre renversé de la paire $(3,4)$ — le swap $R(3,4)$ de GT-21.

La dernière ligne vérifie mécaniquement la cohérence : le `BriserLeTie` du §2.1 appliqué au mur redonne bien **les deux** chambres — A et B — comme les deux faces de la cloison. Le swap de GT-21, redéfini géométriquement, **est** la traversée de ce mur : rien d'autre.

*Fenêtre pédagogique* : sur le mur, l'ensemble des meilleures réponses de Ligne en première colonne **grossit** (de 1 à 2 choix), puis redevient singulier de l'autre côté — avec l'autre choix. Un mur est exactement le lieu où la structure des meilleures réponses **change de cardinal** — la version locale du « mur habité » de GT-21.


### 3.2 La grammaire des swaps relue en longueurs de Coxeter

GT-21 étudiait **six** swaps par côté : $R(a,b)$ pour toutes les paires de valeurs. La géométrie les sépare en deux familles :

- $|a-b| = 1$ (valeurs **adjacentes**) : le swap traverse **un seul mur** — une facette ;
- $|a-b| \geq 2$ : les valeurs sont séparées par d'autres — le swap est un **raccourci** qui enjambe plusieurs murs.

La longueur exacte (nombre de traversées de facettes) d'une transposition de valeurs à distance $d$ est $2d-1$.


In [8]:
// Section 3.2 : les swaps GT-21 en longueurs de Coxeter

int[] AppliquerSwap(int[] t, int a, int b) =>
    t.Select(v => v == a ? b : (v == b ? a : v)).ToArray();

int DistanceCote(int[] t, int[] u)
{
    // Distance dans le graphe des chambres d'un cote (BFS sur les 24 ordres stricts).
    var dLocal = new Dictionary<string, int>();
    var fileLocal = new Queue<string>();
    string ct = string.Join(",", t);
    string cu = string.Join(",", u);
    dLocal[ct] = 0;
    fileLocal.Enqueue(ct);
    while (fileLocal.Count > 0)
    {
        string x = fileLocal.Dequeue();
        if (x == cu) return dLocal[x];
        var xt = x.Split(',').Select(int.Parse).ToArray();
        foreach (int k in new[] { 1, 2, 3 })
        {
            var y = SwapValeursAdjacentes(xt, k);
            string cy = string.Join(",", y);
            if (!dLocal.ContainsKey(cy))
            {
                dLocal[cy] = dLocal[x] + 1;
                fileLocal.Enqueue(cy);
            }
        }
    }
    return -1;
}

// La colonne du milieu est une MESURE -- la distance dans le graphe des chambres
// du 2.2 -- et la colonne de droite une PREDICTION (formule 2d-1). C'est leur
// accord qui fait la preuve ; une colonne calculee par la formule qu'elle
// illustre ne prouverait rien.
Console.WriteLine("Swap    | murs traverses (mesure BFS) | formule 2d-1 | statut");
Console.WriteLine(new string('-', 74));
var swaps = new (int a, int b)[] { (1, 2), (2, 3), (3, 4), (1, 3), (2, 4), (1, 4) };
var mesures = new List<(int a, int b, int mesure)>();
bool longueurUnique = true;
foreach (var (a, b) in swaps)
{
    int d = Math.Abs(a - b);
    var trouvees = new HashSet<int>();
    foreach (var t in chambres) trouvees.Add(DistanceCote(t, AppliquerSwap(t, a, b)));
    longueurUnique = longueurUnique && trouvees.Count == 1;
    int mesure = trouvees.Min();
    mesures.Add((a, b, mesure));
    string statut = d == 1 ? "facette (1 mur)" : "raccourci";
    Console.WriteLine($"R({a},{b})  |            {mesure,2}              |      {2 * d - 1,2}      |  {statut}");
}
Console.WriteLine();
Console.WriteLine("Une seule longueur par swap, quelle que soit la chambre de depart : " + longueurUnique);
bool confirme = mesures.All(m => m.mesure == 2 * Math.Abs(m.a - m.b) - 1);
Console.WriteLine("La mesure confirme la formule 2d-1 sur les six swaps : " + confirme);

// Verification : R(1,3) = R(1,2) puis R(2,3) puis R(1,2)
var t0 = new int[] { 1, 2, 3, 4 };
var direct = AppliquerSwap(t0, 1, 3);
var compose = AppliquerSwap(AppliquerSwap(AppliquerSwap(t0, 1, 2), 2, 3), 1, 2);
Console.WriteLine();
Console.WriteLine("R(1,3) applique a (1,2,3,4)      : " + Format(direct));
Console.WriteLine("R(1,2) puis R(2,3) puis R(1,2)   : " + Format(compose));
Console.WriteLine("Egalite (le raccourci = 3 facettes) : " + direct.SequenceEqual(compose));


Swap    | murs traverses (mesure BFS) | formule 2d-1 | statut


--------------------------------------------------------------------------


R(1,2)  |             1              |       1      |  facette (1 mur)


R(2,3)  |             1              |       1      |  facette (1 mur)


R(3,4)  |             1              |       1      |  facette (1 mur)


R(1,3)  |             3              |       3      |  raccourci


... (3 outputs left out)

La mesure confirme la formule 2d-1 sur les six swaps : True


R(1,3) applique a (1,2,3,4)      : (3, 2, 1, 4)


R(1,2) puis R(2,3) puis R(1,2)   : (3, 2, 1, 4)


Egalite (le raccourci = 3 facettes) : True


**Lecture de la sortie committée** : la grammaire de GT-21 se recompose géométriquement.

La colonne du milieu **mesure** — c'est la distance dans le graphe des chambres du §2.2, parcourue en largeur, et elle ne dépend pas de la chambre de départ ; la colonne de droite **prédit** par la formule $2d-1$. Que les deux coïncident sur les six swaps est le résultat, et non une tautologie : une colonne calculée par la formule qu'elle illustre ne prouverait rien.

- les trois swaps **adjacents** $R(1,2)$, $R(2,3)$, $R(3,4)$ sont les **générateurs** : chacun traverse exactement un mur (une facette), et la mesure le confirme. Avec leurs homologues Colonne, ils engendrent tout le graphe des 576 chambres (§2.2) ;
- les trois **raccourcis** $R(1,3)$, $R(2,4)$, $R(1,4)$ enjambent respectivement 3, 3 et 5 murs — **mesurés** à la même distance, pas déduits de la formule. La décomposition est vérifiée dans la sortie : $R(1,3) = R(1,2) \circ R(2,3) \circ R(1,2)$ — le raccourci et la cascade de facettes aboutissent au **même** 4-uplet, mais la cascade *rencontre* les murs intermédiaires (des jeux à égalités où la structure des meilleures réponses peut basculer), quand le raccourci les saute.

C'est la réponse géométrique à une question laissée ouverte par GT-21 : pourquoi certains swaps « ne comptent-ils » pas (préservation des équilibres) ? Parce qu'un swap n'agit sur les équilibres qu'à travers les murs qu'il traverse qui sont **effectivement habités** (au sens meilleures réponses de GT-21). Les facettes rencontrées par la cascade sont autant d'occasions de bascule ; le raccourci n'échantillonne que l'état final.


## 4. Make tie / break tie : les deux opérations de Bruns-Kimmich

### 4.1 Les opérations, implémentées

La description géométrique s'accompagne de deux opérations duales (rapportées de Bruns-Kimmich, implémentées ici) :

- **make tie** : depuis une chambre, rapprocher deux valeurs adjacentes jusqu'à l'égalité — on **pose** le jeu sur un mur (codim +1) ;
- **break tie** : depuis un mur, séparer l'ex æquo dans un sens ou dans l'autre — on **retombe** dans l'une des deux chambres adjacentes (codim −1).

Ce sont exactement les briques des §2-3, promues au rang d'API.


In [9]:
// Section 4.1 : make_tie / break_tie, operations duales

string FormatJeu((int[] Row, int[] Col) g) => "(" + Format(g.Row) + ", " + Format(g.Col) + ")";

bool SameJeu((int[] Row, int[] Col) a, (int[] Row, int[] Col) b) =>
    a.Row.SequenceEqual(b.Row) && a.Col.SequenceEqual(b.Col);

(int[] Row, int[] Col) MakeTie((int[] Row, int[] Col) jeu, string cote, int k)
{
    // Coalesce les valeurs k et k+1 du cote donne -> jeu sur un mur (codim +1).
    var row = jeu.Row;
    var col = jeu.Col;
    if (cote == "ligne") return (Row: FaireLeTie(row, k), Col: col);
    return (Row: row, Col: FaireLeTie(col, k));
}

(int[] Row, int[] Col) BreakTie((int[] Row, int[] Col) jeu, string cote, string sens)
{
    // Separe l'ex aequo du cote donne, ordre 'haut' ou 'bas' -> chambre (codim -1).
    //
    // La paire liee est replacee dans l'ordre des valeurs DEJA presentes -- celles du
    // dessous, puis la paire, puis celles du dessus -- et le 4-uplet est recanonicalise.
    // Sans ce replacement, separer deux valeurs k/k+1 en laissant les autres en place
    // peut en fabriquer une autre egalite (k+1 figurait deja ailleurs) : le resultat
    // resterait un mur, et l'operation ne ferait pas ce que son nom annonce.
    var row = jeu.Row;
    var col = jeu.Col;
    var t = cote == "ligne" ? row : col;
    var valeursTie = t.GroupBy(x => x).Where(g => g.Count() == 2).Select(g => g.Key).ToArray();
    if (valeursTie.Length == 0) return jeu;   // pas d'ex aequo : operation identite
    int v = valeursTie[0];
    var below = Enumerable.Range(0, 4).Where(p => t[p] < v).OrderBy(p => t[p]).ToArray();
    var above = Enumerable.Range(0, 4).Where(p => t[p] > v).OrderBy(p => t[p]).ToArray();
    var lies = Enumerable.Range(0, 4).Where(p => t[p] == v).ToArray();
    int i = lies[0], j = lies[1];
    var milieu = sens == "bas" ? new[] { j, i } : new[] { i, j };
    var ordre = below.Concat(milieu).Concat(above).ToArray();
    var rangs = new int[4];
    for (int idx = 0; idx < ordre.Length; idx++) rangs[ordre[idx]] = idx + 1;
    if (cote == "ligne") return (Row: rangs, Col: col);
    return (Row: row, Col: rangs);
}

// Demonstration : aller-retour complet depuis le PD
var pd = (Row: new int[] { 3, 1, 4, 2 }, Col: new int[] { 3, 4, 1, 2 });
var mur = MakeTie(pd, "ligne", 3);
var retourHaut = BreakTie(mur, "ligne", "haut");
var retourBas = BreakTie(mur, "ligne", "bas");
Console.WriteLine("Chambre (PD)        : " + FormatJeu(pd));
Console.WriteLine("make_tie(ligne, 3)  : " + FormatJeu(mur) + " (codim " + (Codim(mur.Row) + Codim(mur.Col)) + " )");
Console.WriteLine("break_tie(haut)     : " + FormatJeu(retourHaut) + "-> retrouve le PD : " + SameJeu(retourHaut, pd));
Console.WriteLine("break_tie(bas)      : " + FormatJeu(retourBas) + "-> l'autre chambre (chambre B du 3.1)");
Console.WriteLine();
Console.WriteLine("Le mur porte bien 2 sorties distinctes : " + !SameJeu(retourHaut, retourBas));

// Generalite : un exemple ne prouve rien. Sur les 36 murs simples d'un cote,
// break_tie doit rendre deux CHAMBRES strictes distinctes -- et ce doivent etre
// exactement les deux faces que BriserLeTie (2.1) connait deja a ce mur.
bool deuxChambres = true, memesFaces = true;
var refCol = new int[] { 1, 2, 3, 4 };
foreach (var t in mursSimples)
{
    var haut = BreakTie((Row: t, Col: refCol), "ligne", "haut").Row;
    var bas = BreakTie((Row: t, Col: refCol), "ligne", "bas").Row;
    deuxChambres = deuxChambres && haut.Distinct().Count() == 4 && bas.Distinct().Count() == 4 && !haut.SequenceEqual(bas);
    var (faceA, faceB) = BriserLeTie(t);
    memesFaces = memesFaces && haut.SequenceEqual(faceA) && bas.SequenceEqual(faceB);
}
Console.WriteLine();
Console.WriteLine("Sur les 36 murs simples : deux chambres strictes distinctes = " + deuxChambres);
Console.WriteLine("Sur les 36 murs simples : les deux faces de briser_le_tie  = " + memesFaces);


Chambre (PD)        : ((3, 1, 4, 2), (3, 4, 1, 2))


make_tie(ligne, 3)  : ((3, 1, 3, 2), (3, 4, 1, 2)) (codim 1 )


break_tie(haut)     : ((3, 1, 4, 2), (3, 4, 1, 2))-> retrouve le PD : True


break_tie(bas)      : ((4, 1, 3, 2), (3, 4, 1, 2))-> l'autre chambre (chambre B du 3.1)


Le mur porte bien 2 sorties distinctes : True


Sur les 36 murs simples : deux chambres strictes distinctes = True


Sur les 36 murs simples : les deux faces de briser_le_tie  = True


**Lecture de la sortie committée** : l'aller-retour est **mécaniquement vérifié**. `MakeTie` fait perdre une dimension (le jeu quitte l'ouvert des chambres pour la cloison), `BreakTie` la rend — dans **un des deux sens** : la sortie `haut` retrouve le PD d'origine, la sortie `bas` atterrit dans la chambre B du §3.1. Les deux sorties sont distinctes : c'est la matérialisation calculée du « plusieurs prolongements sont possibles » de la thèse botanique.

Le cas du PD ne vaut pas preuve à lui seul — c'est même le piège de cette opération : sur un mur dont la valeur séparée n'a pas de voisin immédiat déjà placée, séparer « sur place » suffit ; sur les autres, cela fabrique une nouvelle égalité et l'on reste sur un mur. La sortie le tranche sur **les 36 murs simples d'un côté** : chacun rend deux chambres strictes distinctes, et ce sont exactement les deux faces que `BriserLeTie` (§2.1) leur connaissait déjà. L'opération du §4.1 et celle du §2.1 sont donc la même, vue une fois depuis le mur et une fois depuis la chambre.

Un jeu à égalités n'est donc **pas** une spécification incomplète d'un jeu strict : c'est un objet dont le **type** même est différent — un point de l'architecture, avec ses deux faces — et dont le prolongement exige un choix supplémentaire (un sens). Toute la différence entre *assouplir une définition* et *habiter le bord*.


### 4.2 Trois archétypes au bout des murs

Bruns-Kimmich (rapporté) décrivent les jeux 2×2 primitifs comme engendrés depuis des murs par *make/break tie*, sous trois archétypes : **indépendance**, **coordination**, **échange**. Nos implémentations permettent d'en donner des **instanciations calculées** — la définition exacte des archétypes dans la source relève de la dette de sources (§Sources) ; ce qui suit instancie chaque nom par un mur de géométrie naturelle, obtenu par `MakeTie` et vérifié par `BreakTie`.


In [10]:
// Section 4.2 : trois instanciations (RAPPORTEES, cf. Sources)

Console.WriteLine("Archetype       | mur obtenu (Ligne | Colonne)            | geometrie du tie");
Console.WriteLine(new string('-', 90));

// Independance : mur vertical Ligne (r11 = r21) -- Ligne indifferent entre ses
// deux strategies quand Colonne joue a gauche
var baseInd = (Row: new int[] { 1, 3, 2, 4 }, Col: new int[] { 1, 3, 2, 4 });
var murInd = MakeTie(baseInd, "ligne", 1);
Console.WriteLine($"Independance    | {Format(murInd.Row),-18} | {Format(murInd.Col),-18} | vertical (r11 = r21)");

// Coordination : double mur anti-diagonal (r12 = r21 chez les deux joueurs)
var baseCo = (Row: new int[] { 1, 2, 3, 4 }, Col: new int[] { 1, 2, 3, 4 });
var murCo = MakeTie(MakeTie(baseCo, "ligne", 2), "colonne", 2);
Console.WriteLine($"Coordination    | {Format(murCo.Row),-18} | {Format(murCo.Col),-18} | anti-diagonal x2, codim {Codim(murCo.Row) + Codim(murCo.Col)}");

// Echange : mur horizontal Ligne (r11 = r12) -- le gain de Ligne en haut ne
// depend pas de la colonne de Colonne
var baseEch = (Row: new int[] { 1, 2, 3, 4 }, Col: new int[] { 2, 1, 4, 3 });
var murEch = MakeTie(baseEch, "ligne", 1);
Console.WriteLine($"Echange         | {Format(murEch.Row),-18} | {Format(murEch.Col),-18} | horizontal (r11 = r12)");

Console.WriteLine();
Console.WriteLine("Chaque mur se brise dans 2 sens -> chaque archetype borde 2 familles de jeux stricts.");
Console.WriteLine("(Instanciations de notre cru ; la definition source des archetypes : dette, cf. Sources.)");


Archetype       | mur obtenu (Ligne | Colonne)            | geometrie du tie


------------------------------------------------------------------------------------------


Independance    | (1, 2, 1, 3)       | (1, 3, 2, 4)       | vertical (r11 = r21)


Coordination    | (1, 2, 2, 3)       | (1, 2, 2, 3)       | anti-diagonal x2, codim 2


Echange         | (1, 1, 2, 3)       | (2, 1, 4, 3)       | horizontal (r11 = r12)


Chaque mur se brise dans 2 sens -> chaque archetype borde 2 familles de jeux stricts.


(Instanciations de notre cru ; la definition source des archetypes : dette, cf. Sources.)


**Lecture de la sortie committée** : chaque archétype est matérialisé par un jeu mur obtenu **par calcul** depuis un jeu strict :

- **indépendance** — mur *vertical* de Ligne ($r_{11} = r_{21}$) : Ligne exactement indifférent entre ses deux stratégies face à la colonne gauche ; brisé vers le haut ou le bas, ce mur fabrique les jeux où Ligne a une préférence stricte — c'est le voisinage des décisions *unilatérales* ;
- **coordination** — double mur *anti-diagonal* (codim 2, une **arête** de l'architecture) : les deux joueurs lisent leur duel sur la même paire de cases ; brisé dans le même sens des deux côtés → coordination pure, en sens opposés → conflit — le voisinage des jeux de rencontre ;
- **échange** — mur *horizontal* de Ligne ($r_{11} = r_{12}$) : le gain de Ligne en haut ne dépend pas de l'action de Colonne ; brisé, il fabrique les jeux où l'issue dépend du croisement des deux décisions — le voisinage des marchandages.

Ces instanciations **calculées** bornent honnêtement ce que ce notebook établit : les opérations, les comptes, les incidences. La généalogie complète des 2×2 par archétypes — les trois primitives de Bruns-Kimmich — est **rapportée**, non dérivée : elle exige la source primaire (§Sources, dette n°4).


## 5. L'espace des 576 jeux et ses générateurs

La première moitié a décrit l'espace et ses strates ; celle-ci pose la question qui reste : *à quelle distance de swap sont deux jeux, et par quels chemins les relie-t-on ?* La distance se mesure en murs traversés — ce que la §2.2 appelait une facette, et que la §3.1 traversait sur un cas complet.

Deux questions d'analyse d'algorithme s'y ajoutent, qui structurent toute cette moitié :

1. **Le générateur propose** — un parcours en largeur (BFS) explore l'espace niveau par niveau et produit un plus court chemin candidat.
2. **Le certificat garantit** — un module compagnon en Lean (`game_theory_lean/Swaps/Basic.lean`) certifie qu'un chemin proposé est bien formé et mène bien du départ à l'arrivée ; sur le cas témoin Dilemme → Chicken, il certifie aussi qu'aucun chemin plus court n'existe. Le générateur .NET (C#) et le vérificateur Lean restent deux artefacts distincts, chacun à sa place.

Toutes les mesures portent sur l'espace complet des 576 jeux stricts — le quotient par renommage des stratégies (144 classes) est une dette rapportée, jamais affirmée ici.


In [11]:
// Convention GameTheory-3 / GameTheory-21 : une table porte les rangs 1-4
// des quatre cellules dans l'ordre (haut-gauche, haut-droit, bas-gauche,
// bas-droit). Un jeu est le couple (table Ligne, table Colonne).

// Les 24 permutations de (1,2,3,4), en ordre lexicographique.
var PERMS = new List<int[]>();
void GenPerms(List<int> prefix, List<int> rest)
{
    if (rest.Count == 0) { PERMS.Add(prefix.ToArray()); return; }
    for (int i = 0; i < rest.Count; i++)
    {
        var reste = new List<int>(rest);
        int v = reste[i];
        reste.RemoveAt(i);
        prefix.Add(v);
        GenPerms(prefix, reste);
        prefix.RemoveAt(prefix.Count - 1);
    }
}
GenPerms(new List<int>(), new List<int> { 1, 2, 3, 4 });

var JEUX = new List<(int[] Row, int[] Col)>();
foreach (var r in PERMS) foreach (var c in PERMS) JEUX.Add((Row: r, Col: c));   // 576 jeux stricts

// Le generateur elementaire de cette moitie est celui du 2.2, defini une seule
// fois dans ce carnet : vu des valeurs c'est une relabellisation k <-> k+1, vu
// des cellules c'est la traversee du mur ou ces deux rangs coincident.
Func<int[], int, int[]> swapAdjacent = SwapValeursAdjacentes;

List<(int[] Row, int[] Col)> Voisins((int[] Row, int[] Col) g)
{
    // Les six voisins d'un jeu : trois generateurs par joueur.
    var res = new List<(int[] Row, int[] Col)>();
    foreach (int k in new[] { 1, 2, 3 }) res.Add((Row: swapAdjacent(g.Row, k), Col: g.Col));
    foreach (int k in new[] { 1, 2, 3 }) res.Add((Row: g.Row, Col: swapAdjacent(g.Col, k)));
    return res;
}

// Encodages ordinaux canoniques (identiques a GameTheory-3 et GT-21)
var CLASSIC = new Dictionary<string, (int[] Row, int[] Col)>
{
    { "Dilemme", (Row: new int[] { 3, 1, 4, 2 }, Col: new int[] { 3, 4, 1, 2 }) },
    { "Chicken", (Row: new int[] { 3, 2, 4, 1 }, Col: new int[] { 3, 4, 2, 1 }) },
    { "Deadlock", (Row: new int[] { 2, 1, 4, 3 }, Col: new int[] { 2, 4, 1, 3 }) },
    { "Chasse au cerf", (Row: new int[] { 4, 1, 3, 2 }, Col: new int[] { 4, 3, 1, 2 }) },
    { "Harmonie", (Row: new int[] { 4, 3, 2, 1 }, Col: new int[] { 4, 2, 3, 1 }) },
    { "Pennies", (Row: new int[] { 4, 1, 2, 3 }, Col: new int[] { 1, 4, 3, 2 }) }
};

(Dictionary<string, int> dist, Dictionary<string, string> parent) Bfs((int[] Row, int[] Col) depart)
{
    // Parcours en largeur : distances et parents depuis un jeu de depart.
    // Les jeux sont indexes par leur cle textuelle (les tables rang par rang).
    var dist = new Dictionary<string, int>();
    var parent = new Dictionary<string, string>();
    string cleDepart = CleJeu(depart.Row, depart.Col);
    dist[cleDepart] = 0;
    parent[cleDepart] = "";          // racine : aucun parent
    var file = new Queue<(int[] Row, int[] Col)>();
    file.Enqueue(depart);
    while (file.Count > 0)
    {
        var u = file.Dequeue();
        string cu = CleJeu(u.Row, u.Col);
        foreach (var v in Voisins(u))
        {
            string cv = CleJeu(v.Row, v.Col);
            if (!dist.ContainsKey(cv))
            {
                dist[cv] = dist[cu] + 1;
                parent[cv] = cu;
                file.Enqueue(v);
            }
        }
    }
    return (dist, parent);
}

Console.WriteLine($"Espace : {JEUX.Count} jeux stricts, {Voisins(JEUX[0]).Count} generateurs par jeu");
Console.WriteLine("generateur partage avec le 2.2 : " +
    (swapAdjacent.Method == ((Func<int[], int, int[]>)SwapValeursAdjacentes).Method));


Espace : 576 jeux stricts, 6 generateurs par jeu


generateur partage avec le 2.2 : True


## 6. La distance d'un jeu à l'autre

La **distance de swap** entre deux jeux est la longueur du plus court chemin de générateurs qui relie l'un à l'autre — le nombre minimal de murs traversés, au sens du §3.1. C'est la distance de graphe sur les 576 sommets, et le parcours en largeur la mesure exactement : le BFS découvre les jeux par sphères croissantes, et la sphère où un jeu apparaît pour la première fois est sa distance au point de départ.

Lançons le BFS depuis le Dilemme du prisonnier — le jeu le plus commenté de la théorie — et demandons les distances aux cinq autres archétypes canoniques.


In [12]:
// BFS depuis le seul Dilemme -- le moteur Bfs/dist/parent est defini deux cellules
// plus haut avec CLASSIC. DIST_PD : distance de swap (nombre minimal de generateurs)
// de chaque jeu atteint ; PARENT_PD : arbre pour remonter un plus court chemin.
var (DIST_PD, PARENT_PD) = Bfs(CLASSIC["Dilemme"]);

// Connexite mesuree : la BFS epuise tout le graphe -> 576/576 jeux atteints.
Console.WriteLine("jeux atteints depuis le Dilemme : " + DIST_PD.Count + " / 576");

// Excentricite = distance au jeu le plus loin : 12 ici, comme pour chacun des
// 576 jeux -- 12 est le diametre du graphe des chambres.
Console.WriteLine("excentricite du Dilemme : " + DIST_PD.Values.Max());
Console.WriteLine();

Console.WriteLine("Distances depuis le Dilemme :");
// CleJeu(row, col) reconstruit la cle canonique "ligne|colonne" (cellule 16) de
// chaque jeu classique ; les six distances ancrent la lecture : trois cousins a 2
// (Chicken, Deadlock, Chasse au cerf), Harmonie a 6, Pennies a 5.
foreach (var kv in CLASSIC)
{
    Console.WriteLine($"  Dilemme -> {kv.Key,-16} distance = {DIST_PD[CleJeu(kv.Value.Row, kv.Value.Col)]}");
}

jeux atteints depuis le Dilemme : 576 / 576


excentricite du Dilemme : 12


Distances depuis le Dilemme :


  Dilemme -> Dilemme          distance = 0


  Dilemme -> Chicken          distance = 2


  Dilemme -> Deadlock         distance = 2


  Dilemme -> Chasse au cerf   distance = 2


  Dilemme -> Harmonie         distance = 6


  Dilemme -> Pennies          distance = 5


### Interprétation : trois cousins à distance 2, un antagoniste à distance 12

Trois archétypes sont à distance 2 du Dilemme — **Chicken, Deadlock et la chasse au cerf** — et un seul est à distance 12 : son **complément ordinal**, le jeu où chaque rang $r$ est remplacé par $5 - r$ chez les deux joueurs (l'« anti-Dilemme », unique point de la sphère maximale). Entre les deux, Pennies à 5 et Harmonie à 6.

La distance 2 des trois cousins n'est pas un artefact d'encodage : chacun s'obtient du Dilemme en échangeant, **chez chaque joueur**, une paire différente de rangs adjacents :

- **Chicken** : l'ordre des deux *pires* rangs ($P \leftrightarrow S$) — c'est tout ce qui sépare la coopération forcée du Dilemme de l'intimidation réciproque ;
- **Deadlock** : l'ordre des rangs du *milieu* ;
- **la chasse au cerf** : l'ordre des deux *meilleurs* rangs ($T \leftrightarrow R$) — le pari de confiance.

Reconstruisons ces chemins avec la table des parents du BFS.


In [13]:
// Section 6 : nommage des generateurs et reconstruction des chemins

(string Row, string Col) DecodeJeu(string cle)
{
    // La cle d'un jeu est CleJeu(row, col) = "ligne|colonne", chaque table
    // etant la suite de ses rangs separes par des virgules.
    var parties = cle.Split('|');
    return (Row: parties[0], Col: parties[1]);
}

int[] ParseTable(string s) => s.Split(',').Select(x => int.Parse(x)).ToArray();

string NommerEtape(string cleG1, string cleG2)
{
    // Nom du generateur qui fait passer de g1 a son voisin g2.
    var g1 = DecodeJeu(cleG1);
    var g2 = DecodeJeu(cleG2);
    var r1 = ParseTable(g1.Row);
    var c1 = ParseTable(g1.Col);
    var r2 = ParseTable(g2.Row);
    var c2 = ParseTable(g2.Col);
    foreach (int k in new[] { 1, 2, 3 })
    {
        if (SwapValeursAdjacentes(r1, k).SequenceEqual(r2)) return $"R{k}{k + 1}";
        if (SwapValeursAdjacentes(c1, k).SequenceEqual(c2)) return $"C{k}{k + 1}";
    }
    return "?";   // pas voisins
}

List<string> Reconstruire(string cleArrivee, Dictionary<string, string> parent)
{
    // Chemin du depart du BFS jusqu'a l'arrivee, en noms de generateurs.
    var chemin = new List<string>();
    string v = cleArrivee;
    while (parent.TryGetValue(v, out string p) && p.Length > 0)
    {
        chemin.Add(NommerEtape(p, v));
        v = p;
    }
    chemin.Reverse();
    return chemin;
}

(int[] Row, int[] Col) Complement((int[] Row, int[] Col) g)
{
    // Le complement ordinal : chaque rang r devient 5 - r chez les deux joueurs.
    return (Row: g.Row.Select(v => 5 - v).ToArray(), Col: g.Col.Select(v => 5 - v).ToArray());
}

foreach (var nom in new[] { "Chicken", "Deadlock", "Chasse au cerf", "Pennies", "Harmonie" })
{
    var g = CLASSIC[nom];
    string cle = CleJeu(g.Row, g.Col);
    var chemin = Reconstruire(cle, PARENT_PD);
    Console.WriteLine($"Dilemme -> {nom,-16} (distance {DIST_PD[cle],2}) : " + string.Join(" ", chemin));
}

var anti = Complement(CLASSIC["Dilemme"]);
Console.WriteLine();
Console.WriteLine("anti-Dilemme = " + FormatJeu(anti));
Console.WriteLine("distance = " + DIST_PD[CleJeu(anti.Row, anti.Col)] + " ; jeux a cette distance : " +
    DIST_PD.Values.Count(d => d == 12));


Dilemme -> Chicken          (distance  2) : R12 C12


Dilemme -> Deadlock         (distance  2) : R23 C23


Dilemme -> Chasse au cerf   (distance  2) : R34 C34


Dilemme -> Pennies          (distance  5) : R34 R23 C12 C23 C12


Dilemme -> Harmonie         (distance  6) : R12 R34 R23 C12 C34 C23


anti-Dilemme = ((2, 4, 1, 3), (2, 1, 4, 3))


distance = 12 ; jeux a cette distance : 1


### Interprétation : la géographie du Dilemme

Le Dilemme habite un carrefour : ses trois cousins célèbres sont à deux pas, chacun dans une direction cardinale différente (bas, milieu, haut de l'échelle des rangs), et l'unique jeu le plus lointain — à 12 swaps — est son **complément ordinal exact**, celui où chaque cellule qui était un bon rang devient un mauvais. Dans ce jeu antagoniste, la situation du Dilemme est intégralement inversée : ce que la coopération produisait de mieux devient pire. La mesure dit autrement la même chose que le §2.2 : traverser les 12 murs qui séparent un jeu de son complément, c'est renverser toute son échelle de valeurs — et aucun chemin ne peut être plus long, car changer les deux tables complètes exige au plus 6 swaps adjacents chacune (le retournement d'un ordre de 4 éléments).

Le point mérite d'être regardé de plus près : la structure de toutes ces distances est remarquablement régulière, et c'est l'objet du §7.


## 7. La distribution des distances : des sphères emboîtées

Le BFS donne la distance de *chaque* jeu au Dilemme — autrement dit la cardinalité de chaque sphère $S_d = \{G : d(\text{Dilemme}, G) = d\}$ pour $d$ de 0 à 12. Si la géographie du Dilemme était quelconque, ces tailles n'auraient aucune raison d'être régulières. Mesurons.


In [14]:
// Section 7 : distribution des distances depuis le Dilemme

var distribution = DIST_PD.Values.GroupBy(d => d).ToDictionary(g => g.Key, g => g.Count());
int total = distribution.Values.Sum();

Console.WriteLine("d :" + string.Concat(Enumerable.Range(0, 13).Select(d => d.ToString().PadLeft(6))));
Console.WriteLine("N :" + string.Concat(Enumerable.Range(0, 13).Select(d => distribution.TryGetValue(d, out int n) ? n.ToString().PadLeft(6) : "     0")));
Console.WriteLine("total : " + total);
Console.WriteLine();
foreach (int d in Enumerable.Range(0, 13))
{
    int n = distribution.TryGetValue(d, out int v) ? v : 0;
    Console.WriteLine($"  d = {d,2} | {new string('#', n / 2),-48} {n,4}");
}

bool symetrie = Enumerable.Range(0, 13).All(d => (distribution.TryGetValue(d, out int n) ? n : 0) ==
    (distribution.TryGetValue(12 - d, out int m) ? m : 0));
Console.WriteLine();
Console.WriteLine("symetrie d <-> 12 - d : " + symetrie);
Console.WriteLine("maximum atteint en d = 6 : " + (distribution.TryGetValue(6, out int max6) ? max6 : 0) + " jeux");


d :     0     1     2     3     4     5     6     7     8     9    10    11    12


N :     1     6    19    42    71    96   106    96    71    42    19     6     1


total : 576


  d =  0 |                                                     1


  d =  1 | ###                                                 6


  d =  2 | #########                                          19


... (8 outputs left out)

  d = 11 | ###                                                 6


  d = 12 |                                                     1


symetrie d <-> 12 - d : True


maximum atteint en d = 6 : 106 jeux


### Interprétation : une distribution symétrique et factorisée

La distribution est **parfaitement symétrique** autour de 6, en cloche, avec un unique jeu aux deux extrémités ($d = 0$ le Dilemme lui-même, $d = 12$ son complément) et un plateau de 106 jeux à distance 6. Cette régularité appelle une explication structurelle — et elle en a une, calculable de façon indépendante.

La distribution des longueurs de mots sur les 24 permutations d'un côté n'est autre que la distribution des **nombres d'inversions** de $S_4$ : $\{1, 3, 5, 6, 5, 3, 1\}$ — la cellule suivante la mesure, elle ne la recopie pas d'une table. Si la distance d'un jeu à l'autre était *la somme indépendante des distances de chaque côté* — un théorème que le §8 établira par énumération exhaustive — alors la distribution depuis n'importe quel jeu devrait être la **convolution de cette distribution d'un côté avec elle-même**. Vérifions la prédiction *avant* d'établir le théorème qui la justifie.


In [15]:
// Distribution des longueurs d'un côté : les nombres d'inversions sur S4.
// La distance d'une permutation a l'identite (en swaps adjacents de valeurs)
// est son nombre d'inversions -- la longueur de Coxeter.

int Inversions(int[] t)
{
    int c = 0;
    for (int i = 0; i < 4; i++)
        for (int j = i + 1; j < 4; j++)
            if (t[i] > t[j]) c++;
    return c;
}

var distInv = PERMS.GroupBy(Inversions).ToDictionary(g => g.Key, g => g.Count());
var listInv = Enumerable.Range(0, 7).Select(k => distInv.TryGetValue(k, out int n) ? n : 0).ToArray();
Console.WriteLine("distribution des nombres d'inversions sur S4 : [" + string.Join(", ", listInv) + "]" +
    " (somme : " + listInv.Sum() + " permutations )");

// Convolution manuelle : u * v
int[] Convol(int[] u, int[] v)
{
    var w = new int[u.Length + v.Length - 1];
    for (int i = 0; i < u.Length; i++)
        for (int j = 0; j < v.Length; j++)
            w[i + j] += u[i] * v[j];
    return w;
}

var predite = Convol(listInv, listInv);
var mesuree = Enumerable.Range(0, 13).Select(d => distribution.TryGetValue(d, out int n) ? n : 0).ToArray();
Console.WriteLine("convolution (un cote * un cote)  : [" + string.Join(", ", predite) + "]");
Console.WriteLine("distribution mesuree             : [" + string.Join(", ", mesuree) + "]");
Console.WriteLine("coincidence exacte : " + predite.SequenceEqual(mesuree));


distribution des nombres d'inversions sur S4 : [1, 3, 5, 6, 5, 3, 1] (somme : 24 permutations )


convolution (un cote * un cote)  : [1, 6, 19, 42, 71, 96, 106, 96, 71, 42, 19, 6, 1]


distribution mesuree             : [1, 6, 19, 42, 71, 96, 106, 96, 71, 42, 19, 6, 1]


coincidence exacte : True


## 8. Le théorème de décomposition

La coïncidence exacte de la cellule précédente n'est pas un hasard : c'est la trace d'un théorème. Énonçons-le d'abord informellement.

> **Décomposition.** La distance de swap entre deux jeux $G = (r_G, c_G)$ et $H = (r_H, c_H)$ est la **somme des distances de chaque côté** :
> $$d(G, H) = \ell(r_G, r_H) + \ell(c_G, c_H)$$
> où $\ell(a, b)$ est la **longueur de Coxeter** de la relabellisation qui envoie $a$ sur $b$ : le nombre d'inversions de la permutation relative $b \circ a^{-1}$, c'est-à-dire le nombre minimal de transpositions de valeurs adjacentes qui transforment $a$ en $b$.

L'intuition : les générateurs Ligne et Colonne sont *indépendants* — un chemin est un entrelacement d'un mot côté Ligne et d'un mot côté Colonne, et chaque côté se comporte comme le graphe de Cayley de $S_4$ muni des trois transpositions adjacentes, dont la distance est exactement la longueur de Coxeter. Nous n'affirmerons pas cette preuve conceptuelle comme établie (elle est rapportée en dette) — mais nous pouvons faire mieux que la croire : **vérifier l'énoncé sur chacune des 331 776 paires de jeux**, en comparant le BFS (la mesure) à la formule (la prédiction), indépendamment l'une de l'autre.


In [16]:
// Section 8 : longueur de Coxeter d'une relabellisation

int Coxeter(int[] a, int[] b)
{
    // Longueur de Coxeter de la relabellisation a -> b :
    // nombre d'inversions de la permutation relative b . a^{-1}.
    var pos = new int[5];          // pos[v] = position de la valeur v dans a
    for (int i = 0; i < 4; i++) pos[a[i]] = i;
    var pi = new int[4];           // permutation relative
    int idx = 0;
    foreach (int v in new[] { 1, 2, 3, 4 }) pi[idx++] = b[pos[v]];
    int c = 0;
    for (int i = 0; i < 4; i++)
        for (int j = i + 1; j < 4; j++)
            if (pi[i] > pi[j]) c++;
    return c;
}

// Exemples chiffres, un par cote du chemin Dilemme -> Chicken :
Console.WriteLine("Ligne : (3,1,4,2) -> (3,2,4,1), coxeter = " + Coxeter(new int[] { 3, 1, 4, 2 }, new int[] { 3, 2, 4, 1 }));
Console.WriteLine("Colonne : (3,4,1,2) -> (3,4,2,1), coxeter = " + Coxeter(new int[] { 3, 4, 1, 2 }, new int[] { 3, 4, 2, 1 }));
Console.WriteLine("retournement (1,2,3,4) -> (4,3,2,1), coxeter = " + Coxeter(new int[] { 1, 2, 3, 4 }, new int[] { 4, 3, 2, 1 }));
Console.WriteLine("identite (1,2,3,4) -> (1,2,3,4), coxeter = " + Coxeter(new int[] { 1, 2, 3, 4 }, new int[] { 1, 2, 3, 4 }));


Ligne : (3,1,4,2) -> (3,2,4,1), coxeter = 1


Colonne : (3,4,1,2) -> (3,4,2,1), coxeter = 1


retournement (1,2,3,4) -> (4,3,2,1), coxeter = 6


identite (1,2,3,4) -> (1,2,3,4), coxeter = 0


### 8.1 Vérification exhaustive sur les 331 776 paires

Les exemples ci-dessus portent sur trois paires choisies ; l'énoncé, lui, porte sur **toutes**. La vérification est directe et bornée : pour chacune des 576 sources, un parcours en largeur complet donne les distances mesurées, et la formule de Coxeter donne les distances prédites — 576 × 576 = 331 776 paires, dont aucune ne doit diverger.


In [17]:
// Section 8.1 : verification exhaustive de la decomposition

var chrono = System.Diagnostics.Stopwatch.StartNew();

// Verification exhaustive : pour chacune des 576 sources, un BFS complet
// (la mesure), comparee a la formule de Coxeter (la prediction) sur les
// 576 destinations. 576 x 576 = 331 776 paires.
var matrice = new Dictionary<string, Dictionary<string, int>>();
bool coherence = true;
int nbPaires = 0;
foreach (var G in JEUX)
{
    var (dG, _) = Bfs(G);
    matrice[CleJeu(G.Row, G.Col)] = dG;
    foreach (var H in JEUX)
    {
        int formule = Coxeter(G.Row, H.Row) + Coxeter(G.Col, H.Col);
        int mesure = dG[CleJeu(H.Row, H.Col)];
        nbPaires++;
        if (mesure != formule)
        {
            coherence = false;
            Console.WriteLine("CONTRE-EXEMPLE : " + FormatJeu(G) + " -> " + FormatJeu(H) + " " + mesure + " != " + formule);
            break;
        }
    }
    if (!coherence) break;
}
chrono.Stop();
double elapsed = chrono.Elapsed.TotalSeconds;

Console.WriteLine($"paires verifiees : {nbPaires} / 331776 en {elapsed.ToString("0.0", CultureInfo.InvariantCulture)}s");
Console.WriteLine("decomposition d = coxeter(ligne) + coxeter(colonne) : " + coherence);

// Corollaires lus sur la matrice complete :
var excentricites = matrice.Values.Select(d => d.Values.Max()).GroupBy(x => x).ToDictionary(g => g.Key, g => g.Count());
Console.WriteLine("excentricites de tous les jeux : {" + string.Join(", ", excentricites.OrderBy(kv => kv.Key).Select(kv => kv.Key + ": " + kv.Value)) + "}");
var clePD = CleJeu(CLASSIC["Dilemme"].Row, CLASSIC["Dilemme"].Col);
var cleChicken = CleJeu(CLASSIC["Chicken"].Row, CLASSIC["Chicken"].Col);
int spot = matrice[clePD][cleChicken];
Console.WriteLine("spot-check matrice [Dilemme][Chicken] = " + spot + " (BFS §6 : " + DIST_PD[cleChicken] + ")");


paires verifiees : 331776 / 331776 en 4.2s


decomposition d = coxeter(ligne) + coxeter(colonne) : True


excentricites de tous les jeux : {12: 576}


spot-check matrice [Dilemme][Chicken] = 2 (BFS §6 : 2)


### Interprétation : le théorème établi par énumération

**La décomposition vaut pour chacune des 331 776 paires.** Comme le théorème de la condition de transformation dans GT-3d (vérifié sur les 3 456 paires jeu-générateur), l'énoncé est ici *établi par énumération exhaustive* : l'espace est fini, la vérification est complète, et le résultat se lit sur la sortie. La preuve conceptuelle — indépendance des deux côtés, graphe de Cayley de $S_4$, produit cartésien — est **rapportée en dette** (§Sources) : nous n'en livrons pas la démonstration générale, seulement l'évidence exhaustive sur cet espace.

Trois corollaires immédiats, tous mesurés :

- **le diamètre vaut 12** = deux fois le maximum de la longueur de Coxeter d'un côté (6, le retournement) — cohérent avec la mesure du §2.2 ;
- **chaque jeu a une excentricité de 12** : l'espace est parfaitement isotrope, aucun point n'est plus central qu'un autre ; le Dilemme n'occupe aucune position privilégiée — c'est une propriété de *l'espace*, pas du jeu ;
- **la distribution depuis n'importe quel jeu** est la convolution de la distribution d'inversions d'un côté avec elle-même (cellule précédente) — ce qui explique la symétrie parfaite autour de 6.

Un point reste ouvert dans la géographie : si les *distances* sont simples, que dire du **nombre** de plus courts chemins ? C'est l'objet du §9.


## 9. Multiplicités : combien de plus courts chemins

La distance dit *combien de pas* ; elle ne dit pas *par où*. Deux jeux à même distance peuvent être reliés par un nombre très différent de chemins minimaux. Comptons-les en énumérant, pour une cible donnée, tous les chemins qui descendent strictement d'un niveau par étape (tout plus court chemin est une telle descente monotone).


In [18]:
// Section 9 : multiplicites des plus courts chemins

List<(string Cle, string Nom)> VoisinsNommes((int[] Row, int[] Col) g)
{
    // Les six voisins d'un jeu, avec le nom de leur generateur.
    // Chaque element est (cle du voisin, nom du generateur).
    var outp = new List<(string Cle, string Nom)>();
    foreach (int k in new[] { 1, 2, 3 })
    {
        outp.Add((CleJeu(SwapValeursAdjacentes(g.Row, k), g.Col), $"R{k}{k + 1}"));
        outp.Add((CleJeu(g.Row, SwapValeursAdjacentes(g.Col, k)), $"C{k}{k + 1}"));
    }
    return outp;
}

(int[] Row, int[] Col) JeuDepuisCle(string cle)
{
    var d = DecodeJeu(cle);
    return (Row: ParseTable(d.Row), Col: ParseTable(d.Col));
}

List<List<string>> PlusCourtsChemins((int[] Row, int[] Col) depart, (int[] Row, int[] Col) arrivee)
{
    // Enumere tous les plus courts chemins (descentes monotones de niveaux).
    var (d, _) = Bfs(depart);
    string cleDepart = CleJeu(depart.Row, depart.Col);
    string cleArrivee = CleJeu(arrivee.Row, arrivee.Col);
    int n = d[cleArrivee];
    var chemins = new List<(string Cle, List<string> Path)> { (cleDepart, new List<string>()) };
    for (int i = 0; i < n; i++)
    {
        var suivant = new List<(string Cle, List<string> Path)>();
        foreach (var (cle, path) in chemins)
        {
            var g = JeuDepuisCle(cle);
            foreach (var (v, nom) in VoisinsNommes(g))
            {
                if (d.TryGetValue(v, out int dv) && dv == d[cle] + 1)
                {
                    var np = new List<string>(path);
                    np.Add(nom);
                    suivant.Add((v, np));
                }
            }
        }
        chemins = suivant;
    }
    return chemins.Where(c => c.Cle == cleArrivee).Select(c => c.Path).ToList();
}

long Comb(int n, int k)
{
    long r = 1;
    for (int i = 0; i < k; i++) r = r * (n - i) / (i + 1);
    return r;
}

foreach (var nom in new[] { "Chicken", "Deadlock", "Chasse au cerf", "Pennies", "Harmonie" })
{
    var g = CLASSIC[nom];
    var cs = PlusCourtsChemins(CLASSIC["Dilemme"], g);
    Console.WriteLine($"Dilemme -> {nom,-16} distance {DIST_PD[CleJeu(g.Row, g.Col)],2} : " +
        $"{cs.Count,6} plus courts chemins (ex : {string.Join(" ", cs[0])})");
}

var anti = Complement(CLASSIC["Dilemme"]);
int nAnti = PlusCourtsChemins(CLASSIC["Dilemme"], anti).Count;
// Interpretation structurelle du compte (mesuree independamment) :
var identite = (Row: new int[] { 1, 2, 3, 4 }, Col: new int[] { 1, 2, 3, 4 });
var retourneLigne = (Row: new int[] { 4, 3, 2, 1 }, Col: new int[] { 1, 2, 3, 4 });
int motsW0 = PlusCourtsChemins(identite, retourneLigne).Count;
Console.WriteLine();
Console.WriteLine("Dilemme -> anti-Dilemme : " + nAnti + " plus courts chemins");
Console.WriteLine("mots reduits du retournement, un cote : " + motsW0);
Console.WriteLine($"entrelacements C(12,6) x 16 x 16 = {Comb(12, 6)} x {motsW0} x {motsW0} = " +
    (Comb(12, 6) * motsW0 * motsW0));


Dilemme -> Chicken          distance  2 :      2 plus courts chemins (ex : R12 C12)


Dilemme -> Deadlock         distance  2 :      2 plus courts chemins (ex : R23 C23)


Dilemme -> Chasse au cerf   distance  2 :      2 plus courts chemins (ex : R34 C34)


Dilemme -> Pennies          distance  5 :     20 plus courts chemins (ex : C12 C23 C12 R34 R23)


Dilemme -> Harmonie         distance  6 :     80 plus courts chemins (ex : R12 C12 R34 R23 C34 C23)


Dilemme -> anti-Dilemme : 236544 plus courts chemins


mots reduits du retournement, un cote : 16


entrelacements C(12,6) x 16 x 16 = 924 x 16 x 16 = 236544


### Interprétation : de la commutation à l'explosion combinatoire

Le spectre des multiplicités est vaste : **exactement 2** chemins minimaux vers chacun des trois cousins (les deux étapes — une par côté — commutent, donc les deux ordres conviennent, et rien d'autre), **20** vers Pennies, **80** vers l'Harmonie, et **236 544** vers l'anti-Dilemme. Ce dernier compte se factorise exactement : $\binom{12}{6} \times 16 \times 16$ — les 16 *mots réduits* du retournement d'un côté (mesurés par la même énumération, restreinte à un seul côté), au carré pour les deux côtés, entrelacés dans $\binom{12}{6}$ ordres possibles. Le chemin minimal vers l'antagoniste absolu n'est donc pas une rareté : il est abondant, presque *tout chemin glouton qui descend monotone y conduit*.

La leçon structurelle : la distance (§6) est une fonction simple et additive, mais la **géométrie fine** des géodésiques porte l'information combinatoire — le nombre de mots réduits d'un élément de $S_4$ est une donnée non triviale de la théorie des groupes de Coxeter, rapportée en dette comme le reste du vocabulaire. C'est précisément quand les objets deviennent abondants et faciles à générer que la question du *certificat* devient pressante : qui garantit qu'un chemin proposé est correct ? C'est le §10.


## 10. Générateur contre certificat

Récapitulons les deux rôles, qui resteront séparés jusqu'au bout :

- le **générateur** (.NET / C#, ce notebook) *explore* : BFS, énumération des chemins, comptages. Il **propose** des chemins minimaux — mais un programme qui s'exécute n'est pas une preuve qu'il calcule ce qu'il prétend calculer ;
- le **certificat** (Lean, `game_theory_lean/Swaps/Basic.lean`, écrit pour ce grain) **garantit** : qu'un chemin est bien formé (chaque étape est l'un des six générateurs — c'est *structurel*, le type des étapes ne contient rien d'autre), qu'il mène bien du départ à l'arrivée (évalué par le noyau Lean, `rfl`), et — sur le cas témoin — qu'aucun chemin plus court n'existe (énumération décidable).

Commençons par le miroir .NET du vérificateur, pour voir *ce qui se joue* : un chemin **valide** (il arrive) n'est pas un chemin **minimal** (il arrive le plus vite possible).


In [19]:
// Section 10 : miroir .NET du verificateur Lean (Swaps/Basic.lean)

(int[] Row, int[] Col) AppliqueEtape((int[] Row, int[] Col) g, string e)
{
    // Miroir .NET d'appliqueEtape (Swaps/Basic.lean) : applique l'etape nommee e.
    int k = int.Parse(e.Substring(1, 1));
    return e[0] == 'R'
        ? (Row: SwapValeursAdjacentes(g.Row, k), Col: g.Col)
        : (Row: g.Row, Col: SwapValeursAdjacentes(g.Col, k));
}

(int[] Row, int[] Col) AppliqueChemin((int[] Row, int[] Col) g, IEnumerable<string> chemin)
{
    // Miroir .NET d'applique : application successive des etapes.
    var courant = g;
    foreach (string e in chemin) courant = AppliqueEtape(courant, e);
    return courant;
}

bool CheminValide((int[] Row, int[] Col) depart, (int[] Row, int[] Col) arrivee, IEnumerable<string> chemin)
{
    // Certificat d'arrivee : le chemin mene bien de depart a arrivee.
    // Ne dit RIEN de la minimalite.
    var g = AppliqueChemin(depart, chemin);
    return g.Row.SequenceEqual(arrivee.Row) && g.Col.SequenceEqual(arrivee.Col);
}

var PD = CLASSIC["Dilemme"];
var CHICKEN = CLASSIC["Chicken"];
var cMinimal = new[] { "R12", "C12" };                    // le chemin du BFS (§6)
var cDetour = new[] { "R12", "R23", "R23", "C12" };       // un detour : R23 o R23 = identite

Console.WriteLine("chemin minimal  [" + string.Join(", ", cMinimal) + "] : valide = " + CheminValide(PD, CHICKEN, cMinimal));
Console.WriteLine("chemin detour   [" + string.Join(", ", cDetour) + "] : valide = " + CheminValide(PD, CHICKEN, cDetour));
Console.WriteLine();
Console.WriteLine("longueurs : " + cMinimal.Length + " vs " + cDetour.Length +
    " | distance BFS : " + DIST_PD[CleJeu(CHICKEN.Row, CHICKEN.Col)]);


chemin minimal  [R12, C12] : valide = True


chemin detour   [R12, R23, R23, C12] : valide = True


longueurs : 2 vs 4 | distance BFS : 2


### Interprétation : valide n'est pas minimal

Le certificat d'arrivée accepte les **deux** chemins — et il a raison : les deux mènent bien du Dilemme à Chicken (le détour insère un aller-retour $R_{23} \circ R_{23}$, qui s'annule). Mais seul le premier est minimal : la longueur 4 du détour dépasse la distance 2. Un vérificateur d'arrivée est donc *complet mais laxiste* ; discriminer les chemins minimaux exige soit l'exploration exhaustive (ce que fait le BFS), soit une borne inférieure (ce que fait le module Lean sur le cas témoin : aucun chemin de longueur < 2 n'existe, par énumération décidable des cas).

C'est la division du travail que le grain installe : **le générateur produit des candidats abondants** (§9 : des centaines de milliers vers l'anti-Dilemme), **le certificat tranche** — arrivée garantie toujours, minimalité garantie sur le témoin. Voyons le module Lean lui-même, cité aux lignes mesurées du fichier.


## 10-bis. Le module Swaps/Basic.lean, cité aux lignes mesurées

Le module compagnon vit dans `game_theory_lean/Swaps/Basic.lean` (même lake que les preuves Arrow et Shapley). Il est **volontairement sans Mathlib** : tout y est calcul fini décidable sur des listes littérales — les théorèmes se closent par `rfl` (évaluation par le noyau) ou `decide` (énumération décidable), sans tactique sophistiquée ni axiome. La cellule suivante lit le fichier depuis ce clone et imprime chaque définition et chaque théorème à sa ligne réelle — les citations du reste de la section s'appuient sur ces numéros, pas sur une mémoire.


In [20]:
// Section 10-bis : lecture du module game_theory_lean/Swaps/Basic.lean
// (System.IO -- aucune dependance externe, aucun appel reseau)
using System.IO;
using System.Text;

string TrouverSwapsLean()
{
    // Localise Swaps/Basic.lean depuis le repertoire courant ou ses parents
    // (le notebook vit dans GameTheory/, le module dans game_theory_lean/).
    var racines = new List<string>();
    string dir = Directory.GetCurrentDirectory();
    while (!string.IsNullOrEmpty(dir))
    {
        racines.Add(dir);
        var parent = Directory.GetParent(dir);
        if (parent == null) break;
        dir = parent.FullName;
    }
    string[] rels = new[]
    {
        Path.Combine("game_theory_lean", "Swaps", "Basic.lean"),
        Path.Combine("MyIA.AI.Notebooks", "GameTheory", "game_theory_lean", "Swaps", "Basic.lean")
    };
    foreach (var r in racines)
        foreach (var rel in rels)
        {
            string cible = Path.Combine(r, rel);
            if (File.Exists(cible)) return cible;
        }
    return "";
}

string cheminModule = TrouverSwapsLean();
if (cheminModule.Length == 0)
{
    Console.WriteLine("module Swaps/Basic.lean introuvable depuis ce repertoire —" +
        " cloner le depot complet pour la lecture des citations");
}
else
{
    string src = File.ReadAllText(cheminModule, Encoding.UTF8);
    char lf = (char)10;
    char cr = (char)13;
    string srcNorm = src.Replace(cr.ToString() + lf.ToString(), lf.ToString());
    // Le fichier finit par un saut de ligne : la derniere entree vide du Split n'est pas une ligne.
    var lignes = srcNorm.Split(lf);
    if (lignes.Length > 0 && lignes[lignes.Length - 1].Length == 0)
    {
        lignes = lignes.Take(lignes.Length - 1).ToArray();
    }
    Console.WriteLine(Path.GetFileName(cheminModule) + " : " + lignes.Length + " lignes");
    Console.WriteLine();
    for (int i = 0; i < lignes.Length; i++)
    {
        string l = lignes[i];
        string debut = l.TrimStart();
        if (debut.StartsWith("def ") || debut.StartsWith("theorem ") ||
            debut.StartsWith("inductive ") || debut.StartsWith("structure "))
        {
            string extrait = l.Length > 100 ? l.Substring(0, 100) : l;
            Console.WriteLine($"  {i + 1,4}: {extrait.TrimEnd()}");
        }
    }
    Console.WriteLine();
    int nbSorry = src.Split("sorry").Length - 1;
    Console.WriteLine("occurrences de 'sorry' dans le module : " + nbSorry);
}


Basic.lean : 135 lignes

    34: def Table : Type := List Nat
    37: def Jeu : Type := Table × Table
    46: def swapAdj (t : Table) (k : Nat) : Table :=
    50: inductive Etape where
    56: def Etape.k : Etape → Nat
    62: def Etape.ligne : Etape → Bool
    67: def appliqueEtape (g : Jeu) (e : Etape) : Jeu :=
    71: def applique : Jeu → List Etape → Jeu
    78: def dilemme : Jeu := (([3, 1, 4, 2] : Table), [3, 4, 1, 2])
    81: def chicken : Jeu := (([3, 2, 4, 1] : Table), [3, 4, 2, 1])
    84: def cerf : Jeu := (([4, 1, 3, 2] : Table), [4, 3, 1, 2])
    89: def cheminDilemmeChicken : List Etape := [.R12, .C12]
    95: theorem certificat_chemin : applique dilemme cheminDilemmeChicken = chicken := by rfl
   101: theorem chemin_valide_non_minimal :
   106: theorem aucun_chemin_court :
   120: theorem distance_dilemme_chicken :
   130: def cheminValide (depart arrivee : Jeu) (p : List Etape) : Prop :=
   135: theorem certificat_cerf : cheminValide dilemme cerf [.R34, .C34] := by rfl


### Interprétation : ce que chaque théorème garantit

La lecture du module aligne les garanties, dans l'ordre où le notebook les a rencontrées :

- **`certificat_chemin`** — `applique dilemme cheminDilemmeChicken = chicken`, clos par `rfl` : le chemin `[R12, C12]` proposé par le BFS mène bien du Dilemme à Chicken. C'est le certificat d'arrivée, garanti par évaluation du noyau Lean — pas par la ré-exécution d'un programme .NET, mais par la réduction du terme lui-même.
- **`chemin_valide_non_minimal`** — le détour `[R12, R23, R23, C12]` arrive aussi (la cellule miroir l'a mesuré côté .NET) : le module Lean l'admet explicitement, pour montrer que le certificat d'arrivée n'exclut pas les détours.
- **`aucun_chemin_court`** — aucun chemin de longueur 0 ou 1 ne relie le Dilemme à Chicken : longueur 0 rejetée par `decide` (les jeux diffèrent), longueur 1 par `cases e <;> decide` — les six générateurs sont énumérés un à un, chacun tranché par décision. C'est la **borne inférieure**.
- **`distance_dilemme_chicken`** — la conjonction : un chemin de longueur 2 existe *et* aucun plus court n'existe. La distance vaut **exactement 2** — énoncé complet, prouvé.

La minimalité générale (pour deux jeux quelconques) reste du côté du générateur : elle exige l'exploration. Le module Lean la certifie sur le témoin parce que l'espace des chemins de longueur < 2 est assez petit pour l'énumération décidable — c'est exactement la frontière où le certificat devient abordable, et le grain la marque explicitement plutôt que de la masquer. Le build du module (`lake build Swaps`) est sans dépendance Mathlib : il se compile en secondes, aucune preuve du lake n'est touchée.


## Exercices


### Exercice 1 — Énumérer les murs par codimension

Reproduisez et étendez le comptage du §1 : énumérez les 75 ordres faibles, comptez-les par codimension, puis **par type de blocs** (tailles décroissantes). Vérifiez que la somme des comptes par type redonne 75, et que chaque type de mur $(2,1,1)$, $(2,2)$, $(3,1)$, $(4)$ est non vide.

**Indice** : la signature de blocs s'obtient avec `GroupBy` ; la codimension est `4 - t.Distinct().Count()`. Pour le compte par type, regroupez par la signature des tailles de blocs triées dans l'ordre décroissant.

**Étape 1** : énumérer les 75 clés canoniques (§1.1). **Étape 2** : grouper par signature. **Étape 3** : afficher le compte par signature et vérifier la somme.


In [21]:
// Exercice 1 : enumeration des murs par codimension et par type de blocs
// TODO etudiant : reproduire la table du 1.2 (75 ordres faibles, 4 signatures de murs)
// Indice : signature = tailles des blocs triees dans l'ordre decroissant
var resultat_ex1 = new Dictionary<string, int>();  // TODO etudiant : {signature: nombre d'ordres faibles}
Console.WriteLine("Exercice 1 : a completer (compte par type de blocs)");


Exercice 1 : a completer (compte par type de blocs)


### Exercice 2 — Traverser : exhiber un triplet complet

Choisissez un jeu strict qui n'est **pas** le PD (par exemple le jeu de la poule mouillée — vérifiez son encodage ordinal avant de l'utiliser), une de ses facettes, et exhibez le triplet **(chambre A, mur, chambre B)** complet avec les trois matrices, comme au §3.1.

**Indice** : une facette = une paire de valeurs adjacentes $(k, k+1)$ ; le mur s'obtient par `FaireLeTie(table, k)` ; la chambre B par `SwapValeursAdjacentes(table, k)` ; vérifiez enfin que `BriserLeTie(mur)` contient bien les deux chambres.

**Étape 1** : poser les tables du jeu choisi. **Étape 2** : choisir $k$ et construire le mur et la chambre B. **Étape 3** : vérifier la double incidence avec `BriserLeTie`.


In [22]:
// Exercice 2 : exhiber chambre -> mur -> chambre pour un autre jeu
// TODO etudiant : remplir les trois matrices et la verification
(int[] Row, int[] Col)? chambre_A = null;  // TODO etudiant : (row, col) du jeu strict choisi
(int[] Row, int[] Col)? mur_ex2 = null;     // TODO etudiant : le mur traverse (via FaireLeTie)
(int[] Row, int[] Col)? chambre_B = null;   // TODO etudiant : la chambre d'arrivee (via SwapValeursAdjacentes)
Console.WriteLine("Exercice 2 : a completer (triplet chambre-mur-chambre)");


Exercice 2 : a completer (triplet chambre-mur-chambre)


### Exercice 3 — Make tie / break tie : vers les archétypes

Depuis le jeu **Stag Hunt** de GT-21 (Ligne $(4,1,3,2)$, Colonne $(4,3,1,2)$), construisez par `MakeTie` un mur de chaque type : un mur Ligne, un mur Colonne, puis un double mur (codim 2). Pour chacun, appliquez `BreakTie` dans les deux sens et observez les deux chambres de sortie — vous venez de cartographier le voisinage d'un jeu de coordination.

**Indice** : le double mur s'obtient en composant deux `MakeTie` successifs (comme au §4.2 pour la coordination). Les deux sens de `BreakTie` doivent redonner deux jeux **stricts** distincts.

**Étape 1** : construire les trois murs. **Étape 2** : pour chacun, briser dans les deux sens. **Étape 3** : vérifier que chaque paire de sorties est distincte et stricte.


In [23]:
// Exercice 3 : voisinage de Stag Hunt par make_tie / break_tie
// TODO etudiant : les trois murs (ligne, colonne, double) et leurs sorties
Dictionary<string, (int[] Row, int[] Col)> murs_ex3 = null!;         // TODO etudiant : {nom_du_mur: jeu_mur}
Dictionary<string, ((int[] Row, int[] Col) Haut, (int[] Row, int[] Col) Bas)> sorties_ex3 = null!;  // TODO etudiant : {nom_du_mur: (sortie_haut, sortie_bas)}
Console.WriteLine("Exercice 3 : a completer (voisinage de Stag Hunt)");


Exercice 3 : a completer (voisinage de Stag Hunt)


### Exercice 4 — Distance par sphères, sans reconstruire tout le BFS

La fonction `Bfs` du §6 calcule les distances vers *tous* les jeux. Pour une seule cible, c'est du gaspillage — et parfois impossible si l'espace était plus grand. Écrivez `DistanceParSpheres(depart, arrivee)` qui n'explore que les sphères successives jusqu'à rencontrer l'arrivée : maintenez l'ensemble des jeux du niveau courant, engendrez le niveau suivant par générateurs, arrêtez dès que l'arrivée apparaît, et renvoyez la distance atteinte. La fonction doit renvoyer le même verdict que le BFS complet pour toute cible — testez-la au moins sur Chicken (distance 2) et sur l'anti-Dilemme (distance 12).


In [24]:
// Exercice 4 : distance de swap par exploration des sphères successives seulement
int DistanceParSpheres((int[] Row, int[] Col) depart, (int[] Row, int[] Col) arrivee)
{
    // Distance de swap par exploration des sphères successives seulement.
    // Etape 1 : niveau courant = [depart], distance = 0
    // Etape 2 : engendrer le niveau suivant par les six generateurs
    // Etape 3 : si l'arrivee y figure, renvoyer la distance ; sinon recommencer
    // Valeur de retour neutre tant que l'exercice n'est pas fait.
    // TODO etudiant
    return -1;  // a remplacer
}

// Appel de verification (aucun resultat n'est affiche ici : la reponse reste
// a calculer par l'etudiant, puis a comparer avec la distance du BFS §6).
int resultat_ex4 = DistanceParSpheres(CLASSIC["Dilemme"], CLASSIC["Chicken"]);


### Exercice 5 — Le diamètre depuis une autre source : le chiffre, pas l'impression

Le §8 a mesuré que *tous* les jeux ont une excentricité de 12. Vérifiez-le depuis une source différente du Dilemme : écrivez `ExcentriciteDepuis(source)` (distance maximale de la source à tout autre jeu) et appliquez-la à **Pennies**. Puis identifiez **combien de jeux** sont à distance 12 de Pennies et lequel c'est — la réponse est un couple (chiffre, jeu), pas une impression.


In [25]:
// Exercice 5 : excentricite d'un jeu, mesuree depuis une source donnee
int ExcentriciteDepuis((int[] Row, int[] Col) source)
{
    // Excentricite d'un jeu : sa distance maximale a tout autre jeu.
    // Etape 1 : BFS complet depuis source
    // Etape 2 : renvoyer le maximum des distances
    // Valeur de retour neutre tant que l'exercice n'est pas fait.
    // TODO etudiant
    return -1;  // a remplacer
}

// Appel de verification (aucun resultat n'est affiche ici : le chiffre et le jeu
// restent a calculer par l'etudiant, puis a comparer avec le §8).
int resultat_ex5 = ExcentriciteDepuis(CLASSIC["Pennies"]);


### Exercice 6 — Le certificat d'un chemin fourni : valide, et minimal ?

On vous fournit un chemin du Dilemme à Chicken : `["R12", "R23", "R23", "C12"]`. Écrivez `Certificat(depart, arrivee, chemin)` qui renvoie un **couple de booléens** : (validité, minimalité).

- la **validité** se certifie en appliquant le chemin (§10, miroir .NET) et en comparant l'arrivée — c'est ce que `Swaps/Basic.lean` garantit par `rfl` ;
- la **minimalité** exige de comparer la longueur du chemin à la distance réelle — soit par re-BFS (exercice 4), soit par la formule de Coxeter (§8).

Votre verdict sur le chemin fourni doit être `(True, False)` : il arrive, mais en 4 pas au lieu de 2. Puis produisez le certificat complet d'un chemin qui mérite les deux `True`.


In [26]:
// Exercice 6 : certificat d'un chemin -- validite et minimalite
var chemin_fourni = new[] { "R12", "R23", "R23", "C12" };   // un detour, fourni

(bool Valide, bool Minimal) Certificat((int[] Row, int[] Col) depart, (int[] Row, int[] Col) arrivee, IEnumerable<string> chemin)
{
    // Verdict d'un chemin : couple (validite, minimalite).
    // Etape 1 : validite -- appliquer le chemin, comparer l'arrivee
    // Etape 2 : minimalite -- la longueur egale-t-elle la distance ?
    // Renvoyer (bool, bool)
    // Valeur de retour neutre tant que l'exercice n'est pas fait.
    // TODO etudiant
    return (false, false);  // a remplacer
}

// Appel de verification (aucun resultat n'est affiche ici : le verdict reste
// a calculer par l'etudiant, puis a comparer avec l'enonce de l'exercice).
var resultat_ex6 = Certificat(CLASSIC["Dilemme"], CLASSIC["Chicken"], chemin_fourni);


## Conclusion

**Ce que la première moitié établit (tout calculé, tout vérifié dans les sorties) :**

1. l'univers des jeux 2×2 **à rangs** compte $75^2 = 5625$ états, dont les 576 jeux stricts ne sont que les chambres ouvertes (10,2 %) — le reste est l'architecture : 1728 murs simples, 1968 strates de codim 2, jusqu'au jeu totalement plat (codim 6, unique) ;
2. l'incidence est exacte : chaque mur simple sépare **exactement 2 chambres**, chaque chambre a **exactement 3 facettes** par côté — et les 1728 arêtes du graphe des chambres coïncident mur pour mur avec les 1728 strates de codim 1 ;
3. le graphe des chambres est **connexe**, de diamètre 6 par côté (12 pour le jeu) — avec la distribution symétrique des distances $\{1,3,5,6,5,3,1\}$ ;
4. la grammaire des swaps de GT-21 se recompose : les 3 swaps adjacents sont des traversées de facettes (1 mur), les 3 autres des raccourcis — 3, 3 et 5 murs **mesurés** dans le graphe, et non lus sur la formule — avec $R(1,3) = R(1,2) \circ R(2,3) \circ R(1,2)$ vérifié ;
5. *make tie* / *break tie* sont implémentées, et l'aller-retour est vérifié sur **les 36 murs simples** d'un côté — deux chambres strictes distinctes, exactement les faces que `BriserLeTie` leur connaît : un jeu à égalités est un objet **biface**, pas une définition relâchée.

**Ce que la seconde moitié établit :**

- **La distance de swap est mesurable exactement** : BFS depuis n'importe quel jeu, sphères emboîtées. Depuis le Dilemme : Chicken, Deadlock et la chasse au cerf à distance 2 (chacun en échangeant une paire différente de rangs adjacents chez chaque joueur), Pennies à 5, Harmonie à 6, l'anti-Dilemme à 12.
- **Théorème de décomposition (établi par énumération exhaustive, 331 776 paires)** : $d(G, H) = \ell(r_G, r_H) + \ell(c_G, c_H)$, la somme des longueurs de Coxeter des permutations relatives. Corollaires mesurés : diamètre 12, excentricité 12 pour *tous* les jeux (isotropie parfaite), distribution depuis tout jeu = convolution de la distribution d'inversions d'un côté avec elle-même.
- **Les multiplicités portent la combinatoire** : de 2 chemins minimaux (commutation des deux côtés) vers chacun des trois cousins à 236 544 = $\binom{12}{6} \times 16^2$ vers l'anti-Dilemme — les géodésiques abondent exactement là où la distance est maximale.
- **Générateur contre certificat** : le BFS .NET propose, le module Lean `Swaps/Basic.lean` garantit — arrivée par `rfl`, minimalité par énumération décidable sur le témoin Dilemme → Chicken (distance exactement 2). Le module est sans Mathlib et sans `sorry` ; la minimalité générale reste du côté du générateur, et la frontière est explicitée.
- **Position dans la série** : les chambres et les murs sont **dans ce carnet** (§1-§4) ; les chemins — la métrique de l'espace entier — en sont la seconde moitié (§6-§10). Le maillon manquant (rapporté en dette) : le quotient par renommage et la structure $S_4 \times S_4$ comme *tore*, dont notre espace complet est le dépliage.

**Ce qui reste ouvert** — les cinq dettes de dérivation, à honorer depuis les sources primaires avant toute affirmation publique : le quotient $576 \to 144$ (facteur exact à re-dériver), l'action du groupe $S_4 \times S_4$, le tore à 37 trous, la preuve conceptuelle de la décomposition, la théorie générale des groupes de Coxeter derrière les mots réduits — et la vérification firsthand des trois arXiv. Ce carnet ne les affirme pas : il les porte comme dette (§Sources).

**La phrase à retenir** : *un objet botanique n'est pas un objet plus flou — c'est un objet de dimension inférieure depuis lequel plusieurs prolongements sont possibles ; et la distance entre deux chambres se mesure en murs traversés.*

*Chantier des jeux 2×2 — voir aussi* : [GT-3 Topology2x2](GameTheory-03-Topology2x2-CSharp.ipynb) (l'espace et la grammaire) · [Dynamics-10 Le Joueur LLM](Dynamics/Dynamics-10-Joueur-LLM-Python.ipynb) (la table périodique appliquée à un LLM) · [GT-3d Deux-Espèces-de-Flèches](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) (quand un swap préserve les équilibres) · [GT-3e Méta-Actions-Tarifées](GameTheory-03e-Meta-Actions-Tarifees-Python.ipynb) (le capstone).


## Sources et dettes

- **Bruns & Kimmich** — la description géométrique (chambres/murs/strates, make/break tie, archétypes) est **rapportée** de leur formalisation des jeux 2×2, non dérivée de la source primaire ici (dette n°4). Tout ce que le carnet affirme chiffré (75, 5625, 1728, 3, 2, 6, 12) est **calculé localement** et reproductible en ré-exécutant.
- **Robinson & Goforth** — le tableau périodique des jeux 2×2 ; ancêtre Rapoport & Guyer. (L'attribution « Gale et Morris » qui circule dans certains transcripts est une hallucination corrigée.)
- **Vocabulaire** : volontairement minimal — chambres, murs, strates, codimension, facettes, distance de swap, générateur, certificat. Le seul terme de groupe employé est celui **que le carnet calcule** : la longueur de Coxeter d'un swap (§3.2) et la somme de ces longueurs qui donne la distance (§8) ; la théorie générale des groupes de Coxeter reste **rapportée** en dette, jamais affirmée.

**Mesuré dans ce carnet** (chaque chiffre est l'output d'une cellule) : les 75 ordres faibles et les 5 625 jeux à rangs par codimension, l'incidence exacte chambre-mur, le graphe des chambres (connexité, diamètre, distribution), les longueurs de Coxeter des six swaps, l'aller-retour *make tie* / *break tie* ; puis les distances aux archétypes, la distribution complète, la vérification exhaustive de la décomposition (331 776 paires), les excentricités, les multiplicités des plus courts chemins, la factorisation $\binom{12}{6} \times 16^2$, le miroir .NET du vérificateur et les citations de `Swaps/Basic.lean` aux lignes réelles.

**Dettes ouvertes (HARD), jamais affirmées** :

1. **Le quotient $576 \to 144$** (classes de jeux à renommage des stratégies près) et la structure $S_4 \times S_4$ comme tore — Robinson et Goforth rapportent cette lecture ; tout ce carnet travaille sur l'espace complet des 576 jeux et n'affirme rien du quotient.
2. **La preuve conceptuelle de la décomposition** (indépendance des côtés, graphe de Cayley, produit cartésien de graphes) — nous en livrons l'énumération exhaustive, pas la démonstration générale.
3. **Le tore à 37 trous** et la topologie du quotient — hors scope, rapporté.
4. **Les références arXiv** (2309.15981, 2102.00053, 1704.02230) — identités vérifiées en première main (API arXiv) : Tohmé & Viglizzo, *A categorical representation of games* · Czechowski & Piliouras, *Poincaré-Bendixson Limit Sets in Multi-Agent Learning* · Hedges, *Coherence for lenses and open games* ; la lecture intégrale des trois reste à faire.
5. **La théorie des mots réduits** (16 mots pour le retournement de $S_4$) — mesurée par énumération ; la théorie générale des groupes de Coxeter qui l'explique est rapportée.

**Module compagnon** : `game_theory_lean/Swaps/Basic.lean` — pur core Lean, sans Mathlib, sans `sorry` ; `lake build Swaps` sans dépendance.

**Encodages** : identiques à GameTheory-3 et GT-3d (tables de rangs 1-4, cellules dans l'ordre haut-gauche, haut-droit, bas-gauche, bas-droit).

**Jumeau** : ce carnet rejoue, en C# / .NET (BCL seule, aucune dépendance), le fil du carnet Python `GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb` — mêmes instances, mêmes valeurs, mêmes exercices.

> Carnet issu de l'EPIC #26534.
